# Quantum PDE-constrained local-volatility calibration — revised experiments

This notebook alone generates the experiments, the figures (PDF and PNG), the tables and the result ZIP.
**Set the configuration, then choose "Run all".** The default profile is `paper`, which runs every experiment.

- `smoke`: a check that the notebook runs. Results at this small scale carry an explicit label.
- `pilot`: a preliminary study. `paper`: three grids, independent data seeds, initialisation seeds and all ablations.
- No market data are included. The synthetic experiments complete with an empty `MARKET_CSV`.
- The hybrid calibration, the small coherent circuit that is actually built and the classical QHD simulation are kept apart.
- The notebook holds no results before it is run.

For long runs, saving checkpoints to Drive is recommended. A GPU is optional; to use one, select a GPU runtime in Colab and set `USE_GPU=True` in the next cell.


In [ ]:
# Runtime setup — run this before importing JAX.
INSTALL_DEPENDENCIES = True
USE_GPU = False  # Set True only after selecting a Colab GPU runtime.
import os, sys, subprocess, importlib.metadata
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
if INSTALL_DEPENDENCIES:
    jax_package = "jax[cuda12]==0.11.1" if USE_GPU else "jax==0.11.1"
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",
                           jax_package,"qiskit==2.5.2","QuantLib==1.43",
                           "numpy>=2.0","scipy>=1.14","pandas>=2.2","matplotlib>=3.8"])
if not USE_GPU:
    os.environ["JAX_PLATFORMS"]="cpu"
print("Dependency setup complete. Start with a fresh runtime if packages were already imported before this cell.")

In [ ]:
# User configuration
PROFILE = "paper"  # "smoke", "pilot", "paper"
MOUNT_DRIVE = False
OUTPUT_ROOT = "/content/cifer_outputs" if os.path.isdir("/content") else "cifer_outputs"
MARKET_CSV = ""  # Example: "/content/market_quotes.csv"
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/cifer_revision_outputs"


## 1. Core model and reproducibility
Forward-normalised prices, fitted stencil, fixed physical quotes, strike interpolation,
LCHS, exact propagation, sparse discrete adjoint, arbitrage audits, and independent reference data.
The original compact-support basis is evaluated exactly; Fourier approximation is a separate diagnostic.


In [ ]:
"""Reproducible local-volatility calibration; embedded verbatim in the notebook.

All prices are forward-normalised. No market observations are bundled.
"""
from pathlib import Path
from functools import lru_cache, partial
import os, sys, json, time, hashlib, platform, importlib.metadata, warnings
import numpy as np
import pandas as pd
import scipy.linalg as sla
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.optimize import minimize, brentq
from scipy.special import ndtr
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.flatten_util import ravel_pytree
import matplotlib
import matplotlib.pyplot as plt

VERSION = "2026-09-05-r2"
TIMES = np.array([182.0 / 365.0, 1.0])
TRAIN_K = [np.array([.7,.85,.95,1.,1.05,1.2,1.4]),
           np.array([.75,.9,1.,1.1,1.3])]
XI_TRUTH = np.array([[.8,.3,.25,.7],[.6,.35,.3,.5]])
PROFILES = {
    "smoke": dict(ns=[3,4], ref_n=7, seeds=[0], noise_seeds=[101],
                  kernels=[16,32,64], lchs_terms=64, maxiter=12, fd_steps=24,
                  pinn_steps=30, pinn_width=12, collocation=48, ablation=False,
                  circuit_n=2, circuit_m=1, circuit_terms=4, qhd_steps=80),
    "pilot": dict(ns=[5,6], ref_n=9, seeds=[0,1], noise_seeds=[101],
                  kernels=[128,256,512,1024], lchs_terms=512, maxiter=120,
                  fd_steps=256, pinn_steps=1500, pinn_width=24, collocation=256,
                  ablation=True, circuit_n=2, circuit_m=1, circuit_terms=4,
                  qhd_steps=600),
    "paper": dict(ns=[5,6,7], ref_n=11, seeds=list(range(5)),
                  noise_seeds=[101,202,303], kernels=[256,512,1024,2048,4096],
                  lchs_terms=2048, maxiter=220, fd_steps=1024,
                  pinn_steps=6000, pinn_width=32, collocation=768,
                  ablation=True, circuit_n=2, circuit_m=1, circuit_terms=4,
                  qhd_steps=1500),
}

def jsonable(v):
    if isinstance(v, dict): return {str(k):jsonable(x) for k,x in v.items()}
    if isinstance(v, (list,tuple)): return [jsonable(x) for x in v]
    if isinstance(v, (np.ndarray,jax.Array)): return jsonable(np.asarray(v).tolist())
    if isinstance(v, np.generic): return v.item()
    if isinstance(v, Path): return str(v)
    return v

def save_json(path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(jsonable(obj),indent=2,allow_nan=True),encoding="utf-8")
    tmp.replace(path)

def configure(profile="paper", output_root="cifer_outputs", market_csv=""):
    if profile not in PROFILES: raise ValueError("profile must be smoke, pilot or paper")
    cfg=dict(PROFILES[profile], profile=profile, version=VERSION, market_csv=market_csv)
    if market_csv:
        cfg["market_input_sha256"]=hashlib.sha256(Path(market_csv).read_bytes()).hexdigest()
    digest=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:12]
    out=Path(output_root)/f"{profile}_{digest}"
    for name in ["figures","tables","checkpoints","raw","diagnostics"]:
        (out/name).mkdir(parents=True,exist_ok=True)
    cfg["out"]=out
    cfg["fingerprint"]=digest
    cfg["backend"]=str(jax.devices())
    versions={m:importlib.metadata.version(m) for m in
              ["numpy","scipy","pandas","matplotlib","jax","qiskit","QuantLib"]}
    save_json(out/"manifest.json",dict(config=cfg,packages=versions,
              python=sys.version,platform=platform.platform(),
              study_status="not_completed",market_status="not_run",
              time_units="ACT/365F years",price_units="C/(discount*forward)"))
    print("Profile:",profile,"| device:",jax.devices(),"| output:",out)
    return cfg

def basis(x, M=4):
    centres=np.array([-.6,-.2,.2,.6]) if M==4 else np.linspace(-.6,.6,M)
    width=.45 if M==4 else 1.8/max(M-1,1)
    z=(np.asarray(x)[None,:]-centres[:,None])/width
    phi=.04*(1+np.cos(np.pi*np.clip(z,-1,1)))*(np.abs(z)<=1)
    return phi

def curvature_matrix(K):
    K=np.asarray(K); dm=np.diff(K)[:-1]; dp=np.diff(K)[1:]; N=len(K)-2
    B=np.zeros((N,N+2))
    ii=np.arange(N)
    B[ii,ii]=2/(dm*(dm+dp)); B[ii,ii+1]=-2/(dm*dp)
    B[ii,ii+2]=2/(dp*(dm+dp))
    return B

@lru_cache(maxsize=32)
def make_grid(n, M=4, xmin=-1.2, xmax=1.2):
    N=2**n; h=(xmax-xmin)/(N+1)
    xe=np.linspace(xmin,xmax,N+2); K=np.exp(xe); x=xe[1:-1]
    kap=2*np.tanh(h/2)/h
    Lt=np.diag(np.full(N,2/h**2))
    Lt+=np.diag(np.full(N-1,-1/h**2+kap/(2*h)),1)
    Lt+=np.diag(np.full(N-1,-1/h**2-kap/(2*h)),-1)
    beta=(1-K[0])/(K[0]-K[-1]); G=-beta*K[-1]+beta*K
    phi=basis(x,M)
    A0=.005*Lt; Am=.5*phi[:,:,None]*Lt[None,:,:]
    cbar=sum(max(0.,-sla.eigvalsh((a+a.T)/2,subset_by_index=[0,0])[0])
             for a in [A0,*Am])
    return dict(n=n,N=N,M=M,h=h,x=x,xe=xe,K=K,G=G,phi=phi,
                Lt=Lt,A0=A0,Am=Am,cbar=float(cbar),
                v0=np.maximum(1-K[1:-1],0)-G[1:-1],B=curvature_matrix(K),
                xmin=xmin,xmax=xmax)

def assemble(g, xi): return g["A0"]+np.einsum("m,mij->ij",np.asarray(xi),g["Am"])
def sigma2(g, xi): return .01+np.asarray(xi)@g["phi"]

def interpolation(K, query):
    """Piecewise linear in strike, never log-strike; no snapping or clipping."""
    query=np.asarray(query)
    if np.any(query<K[0]) or np.any(query>K[-1]):
        raise ValueError("Query outside the declared strike domain")
    left=np.clip(np.searchsorted(K,query,side="right")-1,0,len(K)-2)
    theta=(query-K[left])/(K[left+1]-K[left])
    W=np.zeros((len(query),len(K))); rr=np.arange(len(query))
    W[rr,left]=1-theta; W[rr,left+1]=theta
    return W

def price_vector(g,v): return np.r_[g["G"][0],np.asarray(v)+g["G"][1:-1],g["G"][-1]]

def audit_prices(K,Cs,tol=1e-8):
    """All metrics retain units; density tolerance is derived, not reused blindly."""
    Cs=np.asarray(Cs); dm=np.diff(K)[:-1]; dp=np.diff(K)[1:]
    butterfly=(dm*Cs[:,2:]+dp*Cs[:,:-2])/(dm+dp)-Cs[:,1:-1]
    density=2*butterfly/(dm*dp)
    calendar=np.diff(np.vstack([np.maximum(1-K,0),Cs]),axis=0)
    slope=np.diff(Cs,axis=1)/np.diff(K)
    bounds=np.minimum(Cs-np.maximum(1-K,0),1-Cs)
    return dict(butterfly_min=float(butterfly.min()),density_min=float(density.min()),
                calendar_min=float(calendar.min()),vertical_min=float(np.minimum(-slope,1+slope).min()),
                price_bound_min=float(bounds.min()),
                butterfly_violation=float(max(0,-butterfly.min())),
                density_violation=float(max(0,-density.min())),
                calendar_violation=float(max(0,-calendar.min())),
                bfly_count=int((butterfly < -tol).sum()),
                cal_count=int((calendar < -tol).sum()),
                audit_price_tolerance=tol)

def certificate(g, approx, exact, roundoff=2e-10):
    """Conditional certificate given a measured absolute price error, not a priori LCHS proof."""
    ap=np.asarray(approx); ex=np.asarray(exact)
    delta=np.max(np.abs(ap-ex),axis=1)
    dm=np.diff(g["K"])[:-1]; dp=np.diff(g["K"])[1:]
    b=(dm*ap[:,2:]+dp*ap[:,:-2])/(dm+dp)-ap[:,1:-1]
    den=2*b/(dm*dp)
    cal=np.diff(np.vstack([np.maximum(1-g["K"],0),ap]),axis=0)
    cb=delta+np.r_[0,delta[:-1]]
    ok=(np.all(b>=-2*delta[:,None]-roundoff) and
        np.all(den>=-4*delta[:,None]/(dm*dp)-roundoff*2/(dm*dp)) and
        np.all(cal>=-cb[:,None]-roundoff))
    return dict(price_error_inf=float(delta.max()),butterfly_bound=float(2*delta.max()),
                calendar_bound=float(cb.max()),density_bound=float((4*delta[:,None]/(dm*dp)).max()),
                certificate_pass=bool(ok))

def lchs_weights(kernel="beta", terms=512, cutoff=48., beta=.8):
    """Symmetric midpoint rule makes conjugate pairing exact up to rounding.
    Normalisation enforces the identity at t=0; it does not recover omitted tails.
    """
    k=-cutoff+(np.arange(terms)+.5)*2*cutoff/terms
    f=1/(np.pi*(1+1j*k)) if kernel=="simple" else np.exp(2**beta-(1+1j*k)**beta)/(2*np.pi)
    w=(2*cutoff/terms)*f/(1-1j*k); total=w.sum()
    if abs(total)<1e-8: raise ValueError("Unstable quadrature normalisation")
    return k,w/total,dict(terms=terms,cutoff=cutoff,dk=2*cutoff/terms,
                          normalisation_defect=float(abs(total-1)),alpha_lcu=float(np.abs(w/total).sum()))

@jax.custom_jvp
def hermitian_unitary(X):
    lam,V=jnp.linalg.eigh(X)
    return (V*jnp.exp(-1j*lam)[None,:])@V.conj().T

@hermitian_unitary.defjvp
def _hermitian_unitary_jvp(primals,tangents):
    (X,),(dX,)=primals,tangents
    lam,V=jnp.linalg.eigh(X)
    U=(V*jnp.exp(-1j*lam)[None,:])@V.conj().T
    diff=lam[:,None]-lam[None,:]; avg=(lam[:,None]+lam[None,:])/2
    divided=-1j*jnp.exp(-1j*avg)*jnp.sinc(diff/(2*jnp.pi))
    dU=V@(divided*(V.conj().T@dX@V))@V.conj().T
    return U,dU

def lchs_apply_jax(A,v,dt,k,w,shift,chunk=16):
    As=A+shift*jnp.eye(A.shape[0]); L=(As+As.T)/2; H=-.5j*(As-As.T)
    pad=(-len(k))%chunk
    kk=jnp.pad(jnp.asarray(k),(0,pad)).reshape(-1,chunk)
    ww=jnp.pad(jnp.asarray(w),(0,pad)).reshape(-1,chunk)
    def part(carry,kw):
        kp,wp=kw
        U=jax.vmap(hermitian_unitary)(dt*(H[None,:,:]+kp[:,None,None]*L))
        return carry+jnp.einsum("k,kij,j->i",wp,U,v),None
    val,_=jax.lax.scan(jax.checkpoint(part),jnp.zeros(v.shape,dtype=jnp.complex128),(kk,ww))
    return jnp.exp(shift*dt)*jnp.real(val)

@lru_cache(maxsize=48)
def solver_functions(n,M,method,terms,cutoff,dt,fd_steps=256,xmin=-1.2,xmax=1.2):
    g=make_grid(n,M,xmin,xmax)
    if method=="fd": return None
    A0=jnp.asarray(g["A0"]); Am=jnp.asarray(g["Am"]); shift=g["cbar"]
    if method.startswith("lchs"):
        k,w,_=lchs_weights(method.split("-")[1],terms,cutoff)
    def forward(xi,v):
        A=A0+jnp.einsum("m,mij->ij",xi,Am)
        if method=="expm": return jax.scipy.linalg.expm(-dt*A,max_squarings=24)@v
        return lchs_apply_jax(A,v,dt,k,w,shift)
    def objective(xi,v,W,y,weight,lam):
        residual=(W@forward(xi,v)-y)*weight
        smooth=jnp.diff(xi,n=2)
        return jnp.mean(residual**2)+lam*jnp.mean(smooth**2) if M>2 else jnp.mean(residual**2)
    return jax.jit(forward),jax.jit(jax.value_and_grad(objective))

def fd_value_grad(g,xi,v0,dt,W,y,weight,lam,steps):
    """Backward Euler, one sparse LU, exact discrete adjoint of the implemented loss."""
    A=assemble(g,xi); dh=dt/steps
    fac=spla.splu(sp.eye(g["N"],format="csc")+dh*sp.csc_matrix(A))
    trajectory=[np.asarray(v0)]
    for _ in range(steps): trajectory.append(fac.solve(trajectory[-1]))
    v=trajectory[-1]; r=W@v-y
    value=float(np.mean((weight*r)**2)); adj=2*W.T@(weight**2*r)/len(y)
    grad=np.zeros(g["M"])
    for state in reversed(trajectory[1:]):
        adj=fac.solve(adj,trans="T")
        grad-=dh*.5*(g["phi"]@(adj*(g["Lt"]@state)))
    if g["M"]>2:
        D=np.diff(np.eye(g["M"]),n=2,axis=0); d=D@xi
        value+=lam*np.mean(d*d); grad+=2*lam*D.T@d/len(d)
    return value,grad,v

def normalised_call(k,T,vol):
    if T<=0: return np.maximum(1-np.asarray(k),0)
    s=np.asarray(vol)*np.sqrt(T)
    d1=-np.log(k)/s+s/2
    return ndtr(d1)-np.asarray(k)*ndtr(d1-s)

def implied_vol(price,k,T):
    lo=max(1-k,0)
    if not np.isfinite(price) or price<=lo+1e-12 or price>=1-1e-12: return np.nan
    try: return brentq(lambda v:normalised_call(k,T,v)-price,1e-6,8.,xtol=1e-12)
    except ValueError: return np.nan

def true_sigma2(x,s,truth):
    if truth=="in": return .01+XI_TRUTH[min(s,1)]@basis(np.asarray(x),4)
    return .01+.055*np.exp(-((np.asarray(x)+.1)/.3)**2)+.035/(1+np.exp(4*np.asarray(x)))

@lru_cache(maxsize=12)
def reference_surface(truth,ref_n,times_tuple=tuple(TIMES),xmin=-1.2,xmax=1.2):
    """Fixed fine strike-space Dupire discretisation, rho=1, sparse exponential action."""
    N=2**ref_n; K=np.exp(np.linspace(xmin,xmax,N+2)); B=curvature_matrix(K)
    beta=(1-K[0])/(K[0]-K[-1]); G=beta*(K-K[-1]); v=np.maximum(1-K[1:-1],0)-G[1:-1]
    Cs=[]; previous=0.
    for T in times_tuple:
        stops=([float(TIMES[0])] if previous<TIMES[0]<T else [])+[T]
        for stop in stops:
            s=int(previous>=TIMES[0]-1e-12)
            a=.5*true_sigma2(np.log(K[1:-1]),s,truth)*K[1:-1]**2
            Q=sp.diags(a)@sp.csc_matrix(B[:,1:-1])
            v=spla.expm_multiply((stop-previous)*Q,v,traceA=float((stop-previous)*Q.diagonal().sum()))
            previous=stop
        Cs.append(np.r_[G[0],v+G[1:-1],G[-1]])
    return K,np.asarray(Cs)

def make_dataset(truth,noise_bp,noise_seed,ref_n):
    K,Cs=reference_surface(truth,ref_n); rng=np.random.default_rng(noise_seed)
    quotes=[]
    for s,T in enumerate(TIMES):
        ks=TRAIN_K[s]; exact=interpolation(K,ks)@Cs[s]
        perturb=rng.uniform(-noise_bp*1e-4,noise_bp*1e-4,len(ks))
        mid=exact+perturb; half=np.full(len(ks),noise_bp*1e-4)
        # Do not clip inconsistent noisy observations into the feasible set.
        quotes.append(dict(T=float(T),k=ks,mid=mid,bid=mid-half,ask=mid+half,clean=exact))
    kid=np.linspace(.7,1.4,181)
    tests=[]
    for s,q in enumerate(quotes):
        mask=~np.any(np.isclose(kid[:,None],q["k"][None,:],atol=1e-10,rtol=0),axis=1)
        kh=kid[mask]; tests.append(dict(T=q["T"],k=kh,price=interpolation(K,kh)@Cs[s]))
    return dict(truth=truth,noise_bp=noise_bp,noise_seed=noise_seed,quotes=quotes,tests=tests,
                dataset_id=f"{truth}_bp{noise_bp:g}_d{noise_seed}",kind="synthetic")

def calibrate_model(g,data,method,cfg,seed=0,lam=0.,start=None):
    rng=np.random.default_rng(seed); v=g["v0"].copy(); hats=[]; outputs=[]; traces=[]
    compilation=0.; opt_time=0.; forward_calls=0; previous=0.; status=[]
    for s,q in enumerate(data["quotes"]):
        dt=q["T"]-previous; previous=q["T"]
        Wfull=interpolation(g["K"],q["k"]); W=Wfull[:,1:-1]
        y=q["mid"]-Wfull@g["G"]
        weight=np.ones(len(y))
        if data.get("kind")=="market":
            weight=1/np.maximum((q["ask"]-q["bid"])/2,1e-5)
        z0=np.clip(.5+.15*rng.standard_normal(g["M"]),0,1) if start is None else np.array(start[s])
        args=(jnp.asarray(v),jnp.asarray(W),jnp.asarray(y),jnp.asarray(weight),float(lam))
        if method=="fd":
            valuegrad=lambda z:fd_value_grad(g,z,v,dt,W,y,weight,lam,cfg["fd_steps"])[:2]
            t=time.perf_counter(); valuegrad(z0); warm=time.perf_counter()-t
        else:
            cutoff=64. if method=="lchs-simple" else 48.
            fwd,vg=solver_functions(g["n"],g["M"],method,cfg["lchs_terms"],cutoff,float(dt),
                                     cfg["fd_steps"],g["xmin"],g["xmax"])
            t=time.perf_counter(); jax.block_until_ready(vg(jnp.asarray(z0),*args)); warm=time.perf_counter()-t
            valuegrad=lambda z:tuple(np.asarray(x) for x in vg(jnp.asarray(z),*args))
        compilation+=warm; records=[]
        def objective(z):
            value,gradient=valuegrad(z); records.append(float(value))
            if not np.isfinite(value) or not np.all(np.isfinite(gradient)):
                raise FloatingPointError("Non-finite calibration loss or gradient")
            return float(value),np.asarray(gradient,dtype=float)
        t=time.perf_counter()
        result=minimize(objective,z0,jac=True,bounds=[(0,1)]*g["M"],method="L-BFGS-B",
                        options=dict(maxiter=cfg["maxiter"],ftol=1e-13,gtol=1e-9,maxls=30))
        opt_time+=time.perf_counter()-t; forward_calls+=len(records)
        hats.append(result.x); traces.append(records); status.append(bool(result.success))
        if method=="fd": v=fd_value_grad(g,result.x,v,dt,W,y,weight,lam,cfg["fd_steps"])[2]
        else: v=np.asarray(fwd(jnp.asarray(result.x),jnp.asarray(v)))
        outputs.append(price_vector(g,v))
    exact=[]; v=g["v0"].copy(); previous=0.
    for xi,q in zip(hats,data["quotes"]):
        v=sla.expm(-(q["T"]-previous)*assemble(g,xi))@v; previous=q["T"]
        exact.append(price_vector(g,v))
    return dict(hats=np.array(hats),C=np.array(outputs),C_exact=np.array(exact),
                trace=traces,converged=all(status),slice_converged=status,
                compile_warm_s=compilation,opt_s=opt_time,nfev=forward_calls)

def evaluate_model(g,data,result,method,seed,lam):
    rs=[]; held=[]; ivs=[]; total=0; validiv=0; inside=[]; invalid_obs=0
    Cs=result["C"]
    for s,q in enumerate(data["quotes"]):
        p=(np.asarray(result["quote_predictions"][s]) if "quote_predictions" in result
           else interpolation(g["K"],q["k"])@Cs[s]); rs.extend(p-q["mid"])
        inside.extend((p>=q["bid"]-1e-10)&(p<=q["ask"]+1e-10))
        invalid_obs+=int(((q["mid"]<np.maximum(1-q["k"],0))|(q["mid"]>1)).sum())
        test=data["tests"][s]
        ph=(np.asarray(result["test_predictions"][s]) if "test_predictions" in result
            else interpolation(g["K"],test["k"])@Cs[s])
        held.extend(ph-test["price"])
        for k,mod,ref in zip(test["k"],ph,test["price"]):
            a=implied_vol(mod,k,q["T"]); b=implied_vol(ref,k,q["T"]); total+=1
            if np.isfinite(a) and np.isfinite(b): ivs.append((a-b)**2); validiv+=1
    aud=audit_prices(g["K"],Cs)
    row=dict(dataset=data["dataset_id"],truth=data["truth"],noise_bp=data["noise_bp"],
             noise_seed=data["noise_seed"],method=method,n=g["n"],N=g["N"],M=g["M"],
             seed=seed,lam=lam,rmse_quotes=float(np.sqrt(np.mean(np.square(rs)))),
             rmse_heldout=float(np.sqrt(np.mean(np.square(held)))),
             iv_rmse=float(np.sqrt(np.mean(ivs))) if ivs else np.nan,
             iv_valid=validiv,iv_total=total,bidask_fraction=float(np.mean(inside)),
             invalid_observations=invalid_obs,nfev=result.get("nfev",0),
             compile_warm_s=result.get("compile_warm_s",0),opt_s=result.get("opt_s",0),
             converged=result.get("converged",False),**aud)
    if "C_exact" in result:
        row.update(certificate(g,Cs,result["C_exact"]))
        row.update({"exact_"+k:v for k,v in audit_prices(g["K"],result["C_exact"]).items()})
    if "hats" in result and data["kind"]=="synthetic":
        covered=(g["K"][1:-1]>=.7)&(g["K"][1:-1]<=1.4)
        row["sigma2_linf"]=float(max(np.max(np.abs(sigma2(g,h)-true_sigma2(g["x"],s,data["truth"]))[covered])
                                    for s,h in enumerate(result["hats"])))
    return row


## 2. Baselines and input validation
AH uses QuantLib's native Andreasen–Huge implementation. PINN-Wam is a documented adaptation
with independent IV and local-variance networks; it is not a reproduction claim.
All solver comparisons use the same observations. Failed calls and invalid IVs remain visible.


In [ ]:
"""Benchmark and diagnostic cells. Requires core.py definitions in this namespace."""
import QuantLib as ql

def ah_calibrate(g,data,cfg):
    today=ql.Date(1,1,2025); ql.Settings.instance().evaluationDate=today
    dc=ql.Actual365Fixed(); zero=ql.YieldTermStructureHandle(ql.FlatForward(today,0.,dc))
    quotes=ql.CalibrationSet()
    for q in data["quotes"]:
        days=int(round(q["T"]*365))
        if abs(days/365-q["T"])>1e-9:
            raise ValueError("AH requires ACT/365F maturities, no silent date rounding")
        for k,price in zip(q["k"],q["mid"]):
            vol=implied_vol(price,k,q["T"])
            if not np.isfinite(vol):
                raise ValueError("AH input has no finite Black IV; quotes retained, baseline marked failed")
            option=ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call,float(k)),ql.EuropeanExercise(today+days))
            quotes.push_back((option,ql.SimpleQuote(float(vol))))
    t=time.perf_counter()
    ah=ql.AndreasenHugeVolatilityInterpl(quotes,ql.QuoteHandle(ql.SimpleQuote(1.)),zero,zero,
                ql.AndreasenHugeVolatilityInterpl.Linear,ql.AndreasenHugeVolatilityInterpl.Call,
                max(200,2*g["N"]),float(g["K"][0]*.8),float(g["K"][-1]*1.2))
    error=ah.calibrationError()  # force the native calibration inside the timer
    elapsed=time.perf_counter()-t
    Cs=np.array([[ah.optionPrice(q["T"],float(k),ql.Option.Call) for k in g["K"]]
                 for q in data["quotes"]])
    qp=[[ah.optionPrice(q["T"],float(k),ql.Option.Call) for k in q["k"]] for q in data["quotes"]]
    hp=[[ah.optionPrice(q["T"],float(k),ql.Option.Call) for k in q["k"]] for q in data["tests"]]
    return dict(C=Cs,quote_predictions=qp,test_predictions=hp,
                converged=True,opt_s=elapsed,compile_warm_s=0.,nfev=0,
                native_error=str(error),method_note="QuantLib Andreasen-Huge linear interpolation")

def init_mlp(key,width):
    sizes=[2,width,width,1]; pars=[]
    for a,b in zip(sizes[:-1],sizes[1:]):
        key,sub=jax.random.split(key)
        pars.append((jax.random.normal(sub,(a,b))*np.sqrt(2/(a+b)),jnp.zeros(b)))
    return pars

def mlp(pars,x):
    for w,b in pars[:-1]: x=jnp.tanh(x@w+b)
    return (x@pars[-1][0]+pars[-1][1]).squeeze()

def pinn_calibrate(g,data,cfg,seed=0,adaptive=True):
    """Wam-style adaptation, NOT a numerical reproduction of CIFEr 2024.
    The original alternating SA / auto-balancing schedule is retained. Two
    independent networks make the PDE residual non-tautological. See README.
    """
    key=jax.random.PRNGKey(seed); k1,k2=jax.random.split(key)
    tree=(init_mlp(k1,cfg["pinn_width"]),init_mlp(k2,cfg["pinn_width"]))
    theta,unravel=ravel_pytree(tree)
    Tmax=data["quotes"][-1]["T"]
    kt=np.concatenate([q["k"] for q in data["quotes"]]); tt=np.concatenate([np.full(len(q["k"]),q["T"]) for q in data["quotes"]])
    yt=jnp.array(np.concatenate([q["mid"] for q in data["quotes"]])); train=jnp.array(np.c_[kt,tt])
    quote_weight=jnp.array(np.concatenate([1/np.maximum((q["ask"]-q["bid"])/2,1e-5)
                 if data["kind"]=="market" else np.ones(len(q["k"])) for q in data["quotes"]]))
    rng=np.random.default_rng(919+seed)
    xc=np.exp(rng.uniform(g["xmin"],g["xmax"],cfg["collocation"]))
    tc=rng.uniform(.02*Tmax,Tmax,cfg["collocation"])
    coll=jnp.array(np.c_[xc,tc]); boundary=jnp.array(np.c_[np.tile(g["K"][[0,-1]],24),np.repeat(np.linspace(.02*Tmax,Tmax,24),2)])
    def functions(th,z):
        p,lv=unravel(th); k,t=z
        inp=jnp.array([jnp.log(k)/1.2,2*t/Tmax-1])
        iv=.03+.35*jax.nn.softplus(mlp(p,inp))
        s=iv*jnp.sqrt(jnp.maximum(t,1e-10)); d=-jnp.log(k)/s+s/2
        c=jax.scipy.special.ndtr(d)-k*jax.scipy.special.ndtr(d-s)
        variance=.01+.04*jax.nn.softplus(mlp(lv,inp))
        return c,variance
    def cfun(th,z): return functions(th,z)[0]
    def errors(th):
        pred=jax.vmap(cfun,(None,0))(th,train)
        vals=jax.vmap(functions,(None,0))(th,coll)
        deriv=jax.vmap(jax.grad(cfun,argnums=1),(None,0))(th,coll)
        hess=jax.vmap(jax.hessian(cfun,argnums=1),(None,0))(th,coll)
        ck,ct=deriv[:,0],deriv[:,1]; ckk=hess[:,0,0]
        pde=ct-.5*vals[1]*coll[:,0]**2*ckk
        cb=jax.vmap(cfun,(None,0))(th,boundary)
        btarget=jnp.maximum(1-boundary[:,0],0)
        return (((pred-yt)*quote_weight)**2,pde**2,jax.nn.relu(-ckk)**2,
                jax.nn.relu(-ct)**2,(jax.nn.relu(ck)**2+jax.nn.relu(-1-ck)**2),
                (cb-btarget)**2)
    sa=tuple(jnp.ones(len(x)) for x in errors(theta)); weights=jnp.ones(6)
    def components(th,sa,weights): return jnp.stack([w*jnp.mean(a*e) for w,a,e in zip(weights,sa,errors(th))])
    @jax.jit
    def update(th,m,v,step,sa,weights):
        loss,gr=jax.value_and_grad(lambda p:jnp.sum(components(p,sa,weights)))(th)
        m=.9*m+.1*gr; v=.999*v+.001*gr**2
        lr=1e-3*.9**(step/2000)
        th=th-lr*(m/(1-.9**(step+1)))/(jnp.sqrt(v/(1-.999**(step+1)))+1e-8)
        return th,m,v,loss
    @jax.jit
    def rebalance(th,sa,w):
        jac=jax.jacrev(components)(th,sa,w)
        norms=jnp.mean(jnp.abs(jac),axis=1)
        target=jnp.where(norms>1e-15,jnp.sum(norms)/jnp.maximum(norms,1e-15),1.)
        return jax.lax.stop_gradient(.9*w+.1*jnp.clip(target,1e-3,1e3))
    @jax.jit
    def ascend(th,sa,w):
        # Gradient ascent of weighted mean loss w.r.t. per-point multipliers.
        return tuple(jnp.minimum(a+weight*err/len(err),1e3) for a,weight,err in zip(sa,w,errors(th)))
    m=jnp.zeros_like(theta); v=m; hist=[]
    t=time.perf_counter(); jax.block_until_ready(update(theta,m,v,0,sa,weights))
    if adaptive:
        jax.block_until_ready(rebalance(theta,sa,weights)); jax.block_until_ready(ascend(theta,sa,weights))
    warm=time.perf_counter()-t; t=time.perf_counter()
    for step in range(cfg["pinn_steps"]):
        if adaptive and step%100==0: weights=rebalance(theta,sa,weights)
        if adaptive and (step+50)%100==0: sa=ascend(theta,sa,weights)
        theta,m,v,loss=update(theta,m,v,step,sa,weights)
        if step%100==0 or step==cfg["pinn_steps"]-1:
            comp=np.asarray(components(theta,tuple(jnp.ones_like(a) for a in sa),jnp.ones(6)))
            hist.append(dict(step=step,loss=float(loss),fit=comp[0],pde=comp[1],butterfly=comp[2],calendar=comp[3]))
    jax.block_until_ready(theta); elapsed=time.perf_counter()-t
    if not np.all(np.isfinite(theta)): raise FloatingPointError("PINN non-finite parameters")
    predict=jax.jit(jax.vmap(cfun,(None,0)))
    Cs=np.array([predict(theta,jnp.array(np.c_[g["K"],np.full(len(g["K"]),q["T"])])) for q in data["quotes"]])
    qp=[np.asarray(predict(theta,jnp.array(np.c_[q["k"],np.full(len(q["k"]),q["T"])]))) for q in data["quotes"]]
    hp=[np.asarray(predict(theta,jnp.array(np.c_[q["k"],np.full(len(q["k"]),q["T"])]))) for q in data["tests"]]
    return dict(C=Cs,quote_predictions=qp,test_predictions=hp,
                converged=False,training_completed=True,opt_s=elapsed,compile_warm_s=warm,
                nfev=cfg["pinn_steps"],pinn_history=hist,weights=np.asarray(weights),
                parameter_count=len(theta),method_note="Fixed epoch budget; convergence not asserted")

def run_calibration_suite(cfg):
    allrows=[]; failures=[]; selected={}; out=cfg["out"]
    cases=[(truth,bp,ds) for truth in ["in","out"] for bp in [0.,5.]
           for ds in ([101] if bp==0 else cfg["noise_seeds"])]
    comparison_n=cfg["ns"][min(1,len(cfg["ns"])-1)]
    for truth,bp,ds in cases:
        data=make_dataset(truth,bp,ds,cfg["ref_n"])
        save_json(out/"raw"/(data["dataset_id"]+".json"),data)
        for n in cfg["ns"]:
            g=make_grid(n)
            methods=["expm","fd","lchs-beta","lchs-simple"]
            if n==comparison_n: methods += ["AH","PINN-fixed","PINN-Wam"]
            for method in methods:
                seeds=[cfg["seeds"][0]] if method=="AH" else cfg["seeds"]
                for seed in seeds:
                    lam=0.  # core solver comparison uses exactly the same objective
                    key=f'{data["dataset_id"]}_n{n}_{method}_s{seed}'
                    cp=out/"checkpoints"/(key+".json")
                    if cp.exists():
                        payload=json.loads(cp.read_text()); allrows.append(payload["row"])
                        if seed==cfg["seeds"][0] and n==comparison_n:
                            selected[key]=payload
                        continue
                    fp=out/"checkpoints"/(key+"_failure.json")
                    if fp.exists():
                        failures.append(json.loads(fp.read_text())); continue
                    try:
                        if method=="AH": result=ah_calibrate(g,data,cfg)
                        elif method.startswith("PINN"):
                            result=pinn_calibrate(g,data,cfg,seed,adaptive=(method=="PINN-Wam"))
                        else: result=calibrate_model(g,data,method,cfg,seed,lam)
                        row=evaluate_model(g,data,result,method,seed,lam)
                        row["profile"]=cfg["profile"]; row["kind"]="synthetic"
                        payload=dict(row=row,result=result)
                        save_json(cp,payload); allrows.append(row)
                        if seed==cfg["seeds"][0] and n==comparison_n: selected[key]=jsonable(payload)
                        print(f'{key}: quote {row["rmse_quotes"]:.3e}, test {row["rmse_heldout"]:.3e}, opt {row["opt_s"]:.1f}s')
                    except (ValueError,FloatingPointError,RuntimeError) as exc:
                        failure=dict(key=key,method=method,dataset=data["dataset_id"],reason=str(exc))
                        save_json(fp,failure)
                        failures.append(failure); print("FAILED:",failure)
                    pd.DataFrame(allrows).to_csv(out/"results.csv",index=False)
                    save_json(out/"failures.json",failures)
    pd.DataFrame(allrows).to_csv(out/"results.csv",index=False)
    save_json(out/"failures.json",failures)
    save_json(out/"selected_examples.json",selected)
    return pd.DataFrame(allrows),selected

def finite_difference_grad(fun,x,eps=1e-5):
    eye=np.eye(len(x))*eps
    return np.array([(fun(x+d)-fun(x-d))/(2*eps) for d in eye])

def run_validation(cfg):
    rows=[]
    for n in cfg["ns"]:
        g=make_grid(n); N=g["N"]; h=g["h"]
        rho=4*np.sinh(h/2)**2/h**2
        operator=rho*g["K"][1:-1,None]**2*g["B"]
        assert np.allclose(operator[:,1:-1],-g["Lt"],atol=1e-8,rtol=1e-11)
        lift=float(np.max(np.abs(operator@g["G"])))
        rng=np.random.default_rng(901); worst=0.; positive_shift=None
        for xi in np.r_[rng.uniform(0,1,(16,4)),np.eye(4),1-np.eye(4)]:
            A=assemble(g,xi); c=max(0.,-sla.eigvalsh((A+A.T)/2)[0]); worst=max(worst,c)
            if c>1e-5: positive_shift=xi
            off=-A+np.diag(np.diag(A)); assert np.min(off)>=-1e-12
            density_generator=g["B"][:,1:-1]@np.diag(rho*.5*sigma2(g,xi)*g["K"][1:-1]**2)
            assert np.min(density_generator-np.diag(np.diag(density_generator)))>=-1e-10
            assert c<=g["cbar"]+1e-10
        xi=np.array([.4,.6,.5,.3]); data=make_dataset("in",0.,101,cfg["ref_n"])
        q=data["quotes"][0]; Wf=interpolation(g["K"],q["k"]); W=Wf[:,1:-1]; y=q["mid"]-Wf@g["G"]
        val,grad,_=fd_value_grad(g,xi,g["v0"],q["T"],W,y,np.ones(len(y)),0.,cfg["fd_steps"])
        fd=finite_difference_grad(lambda z:fd_value_grad(g,z,g["v0"],q["T"],W,y,np.ones(len(y)),0.,cfg["fd_steps"])[0],xi)
        fd_error=float(np.linalg.norm(grad-fd)/max(1e-12,np.linalg.norm(fd)))
        assert fd_error<1e-4
        fw,vg=solver_functions(n,4,"lchs-beta",cfg["lchs_terms"],48.,float(q["T"]))
        args=tuple(jnp.array(x) for x in [g["v0"],W,y,np.ones(len(y))])+(0.,)
        _,grad=vg(jnp.array(xi),*args)
        fd=finite_difference_grad(lambda z:float(vg(jnp.array(z),*args)[0]),xi)
        lerr=float(np.linalg.norm(np.asarray(grad)-fd)/max(1e-12,np.linalg.norm(fd)))
        assert lerr<3e-4
        # Constant-vector and lift consistency in strike-space observation.
        assert np.allclose(Wf.sum(1),1) and np.allclose(Wf@g["K"],q["k"])
        rows.append(dict(n=n,lift_residual=lift,fitted_identity_residual=float(np.max(np.abs(operator[:,1:-1]+g["Lt"]))),
                         fd_gradient_relative_error=fd_error,lchs_gradient_relative_error=lerr,
                         maximum_sampled_shift=worst,uniform_shift=g["cbar"],
                         positive_shift_sample=positive_shift is not None))
    # Reference refinement at unchanged quotes/tests, not changing the data per N.
    for truth in ["in","out"]:
        K,C=reference_surface(truth,cfg["ref_n"]); K2,C2=reference_surface(truth,cfg["ref_n"]+1)
        pts=np.linspace(.7,1.4,101)
        err=float(np.max(np.abs(C@interpolation(K,pts).T-C2@interpolation(K2,pts).T)))
        rows.append(dict(test="reference_refinement",truth=truth,reference_price_inf=err))
    frame=pd.DataFrame(rows); frame.to_csv(cfg["out"]/"validation.csv",index=False)
    print("Structural identities, fixed quote operators, and implemented gradients: PASS")
    return frame

def run_error_budget(cfg):
    rows=[]; rng=np.random.default_rng(707)
    for n in cfg["ns"]:
        g=make_grid(n); xi=rng.uniform(.15,.85,(2,4))
        exact=[]; v=g["v0"].copy(); prior=0.
        for s,T in enumerate(TIMES):
            v=sla.expm(-(T-prior)*assemble(g,xi[s]))@v; exact.append(price_vector(g,v)); prior=T
        for kernel in ["simple","beta"]:
            for terms in cfg["kernels"]:
                for cutoff in ([16.,32.,64.] if cfg["profile"]!="smoke" else [8.,24.]):
                    k,w,meta=lchs_weights(kernel,terms,cutoff)
                    v=jnp.array(g["v0"]); approx=[]; previous=0.
                    call=jax.jit(lambda A,v,dt:lchs_apply_jax(A,v,dt,k,w,g["cbar"]))
                    t=time.perf_counter(); jax.block_until_ready(call(jnp.array(assemble(g,xi[0])),v,TIMES[0])); warm=time.perf_counter()-t
                    t=time.perf_counter()
                    for s,T in enumerate(TIMES):
                        v=call(jnp.array(assemble(g,xi[s])),v,T-previous); previous=T
                        approx.append(price_vector(g,v))
                    jax.block_until_ready(v); elapsed=time.perf_counter()-t
                    cert=certificate(g,approx,exact)
                    assert cert["certificate_pass"],"Error-bound check failed"
                    row=dict(n=n,kernel=kernel,**meta,**cert,**audit_prices(g["K"],approx),
                             solver_s=elapsed,compile_warm_s=warm,shift=g["cbar"],
                             simulation_time_proxy=cutoff*sum(TIMES-np.r_[0,TIMES[:-1]])*np.linalg.norm(assemble(g,xi[0]),2))
                    rows.append(row)
        pd.DataFrame(rows).to_csv(cfg["out"]/"error_budget.csv",index=False)
    return pd.DataFrame(rows)

def run_diagnostics(cfg,selected):
    rows=[]; n=cfg["ns"][min(1,len(cfg["ns"])-1)]; g=make_grid(n)
    data=make_dataset("in",0.,101,cfg["ref_n"])
    key=f'{data["dataset_id"]}_n{n}_expm_s{cfg["seeds"][0]}'
    hats=np.asarray(selected[key]["result"]["hats"])
    for bits in [2,4,6,8,10,12]:
        quant=np.round(hats*(2**bits-1))/(2**bits-1)
        exact=[]; v=g["v0"].copy(); previous=0.
        for h,q in zip(quant,data["quotes"]):
            v=sla.expm(-(q["T"]-previous)*assemble(g,h))@v; previous=q["T"]
            exact.append(price_vector(g,v))
        result=dict(hats=quant,C=np.array(exact),C_exact=np.array(exact),converged=True)
        row=evaluate_model(g,data,result,"quantised-expm",0,0.)
        row.update(bits=bits,design_qubits=bits*g["M"],hxi=1/(2**bits-1),
                   inverse_hxi_squared=(2**bits-1)**2,continuous_reference=selected[key]["row"]["rmse_quotes"])
        rows.append(row)
    pd.DataFrame(rows).to_csv(cfg["out"]/"quantisation.csv",index=False)
    # Smoothness: global Fourier approximation of the compactly supported bump.
    x=np.linspace(-1.2,1.2,4096,endpoint=False); phi=basis(x)[1]; ft=np.fft.fft(phi)/len(x)
    freqs=np.fft.fftfreq(len(x))*len(x); fourier=[]
    for degree in [2,4,8,16,32,64,128]:
        approx=np.fft.ifft(ft*(np.abs(freqs)<=degree)*len(x)).real
        fourier.append(dict(degree=degree,error_inf=float(np.max(np.abs(approx-phi))),
                            minimum_basis=float(approx.min()),variance_floor=.01,
                            coefficient_error_budget=float(np.max(np.abs(approx-phi))*4),
                            operator_error_bound=float(2*np.max(np.abs(approx-phi))*np.linalg.norm(g["Lt"],2))))
    pd.DataFrame(fourier).to_csv(cfg["out"]/"fourier_basis.csv",index=False)
    # FD accuracy at one fixed coefficient, separate from optimisation.
    q=data["quotes"][0]; A=assemble(g,hats[0]); ex=sla.expm(-q["T"]*A)@g["v0"]
    fdrows=[]
    for steps in ([8,16,32,64] if cfg["profile"]=="smoke" else [32,64,128,256,512,1024,2048]):
        _,_,v=fd_value_grad(g,hats[0],g["v0"],q["T"],np.eye(g["N"]),np.zeros(g["N"]),np.ones(g["N"]),0.,steps)
        fdrows.append(dict(steps=steps,price_error_inf=float(np.max(np.abs(v-ex)))))
    pd.DataFrame(fdrows).to_csv(cfg["out"]/"fd_convergence.csv",index=False)
    # Quote Jacobian singular values identify unobserved parameter directions.
    jacrows=[]; v=jnp.asarray(g["v0"]); previous=0.
    for s,q in enumerate(data["quotes"]):
        fw,_=solver_functions(n,4,"expm",1,1.,float(q["T"]-previous)); previous=q["T"]
        W=jnp.asarray(interpolation(g["K"],q["k"])[:,1:-1])
        J=np.asarray(jax.jacrev(lambda z:W@fw(z,v))(jnp.array(hats[s])))
        sv=sla.svdvals(J)
        for j,val in enumerate(sv): jacrows.append(dict(slice=s+1,index=j+1,singular_value=val,condition=float(sv[0]/max(sv[-1],1e-15))))
        v=fw(jnp.array(hats[s]),v)
    pd.DataFrame(jacrows).to_csv(cfg["out"]/"identifiability.csv",index=False)
    # Same out-of-span data: M and regularisation are varied independently.
    ab=[]
    if cfg["ablation"]:
        d=make_dataset("out",5.,101,cfg["ref_n"])
        for M in [2,4,8]:
            gm=make_grid(n,M)
            for lam in [0.,1e-7,1e-5,1e-3]:
                for method in ["expm","lchs-beta"]:
                    r=calibrate_model(gm,d,method,cfg,0,lam)
                    ab.append(evaluate_model(gm,d,r,method,0,lam))
                    pd.DataFrame(ab).to_csv(cfg["out"]/"ablation.csv",index=False)
    return pd.DataFrame(rows)

MARKET_COLUMNS=["date","expiry","underlying","strike","forward","discount","bid","ask",
                "option_type","exercise_style","carry_model","source"]

def load_market_csv(path):
    frame=pd.read_csv(path)
    missing=set(MARKET_COLUMNS)-set(frame.columns)
    if missing: raise ValueError("Missing market columns: "+str(sorted(missing)))
    for c in ["date","expiry"]: frame[c]=pd.to_datetime(frame[c],errors="raise")
    for c in ["strike","forward","discount","bid","ask"]:
        frame[c]=pd.to_numeric(frame[c],errors="raise")
        if not np.isfinite(frame[c]).all(): raise ValueError("Non-finite "+c)
    if not (frame.exercise_style.str.lower()=="european").all(): raise ValueError("Only European quotes are supported")
    if not (frame.carry_model=="deterministic_proportional").all():
        raise ValueError("Declare deterministic_proportional carry; discrete cash dividends are unsupported")
    if ((frame[["strike","forward","discount"]]<=0).any().any() or (frame.bid<0).any() or (frame.ask<frame.bid).any()):
        raise ValueError("Invalid prices, discount, forward or crossed bid/ask")
    if frame.duplicated(["date","expiry","underlying","strike"]).any():
        raise ValueError("Duplicate contract/strike rows. Resolve explicitly; do not average silently")
    if frame.source.fillna("").str.strip().eq("").any(): raise ValueError("A data source is required on every row")
    datasets=[]
    for (date,underlying),day in frame.groupby(["date","underlying"]):
        qs=[]; tests=[]
        for expiry,part in day.groupby("expiry"):
            part=part.sort_values("strike").copy()
            T=(expiry-date).days/365.
            if T<=0: raise ValueError("Expired or same-day quotes")
            if len(part)<6: raise ValueError("Need at least six quotes per expiry for held-out evaluation")
            if part.forward.nunique()!=1 or part.discount.nunique()!=1: raise ValueError("Inconsistent forward/discount per expiry")
            norm=part.forward.to_numpy()*part.discount.to_numpy()
            parity=part.discount.to_numpy()*(part.forward.to_numpy()-part.strike.to_numpy())
            cp=part.option_type.str.lower()
            if not cp.isin(["call","put"]).all(): raise ValueError("option_type must be call or put")
            bid=(part.bid.to_numpy()+np.where(cp=="put",parity,0))/norm
            ask=(part.ask.to_numpy()+np.where(cp=="put",parity,0))/norm
            k=part.strike.to_numpy()/part.forward.to_numpy(); mid=(bid+ask)/2
            if k.min()<np.exp(-1.2) or k.max()>np.exp(1.2): raise ValueError("Market strikes exceed the supported domain")
            held=np.arange(len(k))%4==1  # deterministic split before any fitting
            qs.append(dict(T=T,k=k[~held],mid=mid[~held],bid=bid[~held],ask=ask[~held]))
            tests.append(dict(T=T,k=k[held],price=mid[held],bid=bid[held],ask=ask[held]))
        if len(qs)<2: raise ValueError("Need at least two expiries per market date")
        ident=hashlib.sha256(f"{date}-{underlying}".encode()).hexdigest()[:12]
        datasets.append(dict(dataset_id="market_"+ident,kind="market",truth="market",noise_bp=0.,noise_seed=0,
                             date=str(date.date()),underlying=str(underlying),quotes=qs,tests=tests))
    return datasets

def run_market(cfg):
    path=cfg.get("market_csv",""); out=cfg["out"]
    if not path:
        pd.DataFrame(columns=MARKET_COLUMNS).to_csv(out/"market_template.csv",index=False)
        save_json(out/"market_status.json",dict(status="not_run",reason="No real market CSV supplied; template has headers only"))
        print("Market study: NOT RUN. Set MARKET_CSV to a bid-ask CSV. No synthetic substitute is used.")
        return pd.DataFrame()
    datasets=load_market_csv(path); g=make_grid(cfg["ns"][min(1,len(cfg["ns"])-1)])
    rows=[]; failed=[]
    for d in datasets:
        save_json(out/"raw"/(d["dataset_id"]+".json"),d)
        for method in ["expm","fd","lchs-beta","AH","PINN-Wam"]:
            try:
                if method=="AH": r=ah_calibrate(g,d,cfg)
                elif method=="PINN-Wam": r=pinn_calibrate(g,d,cfg,0,True)
                else: r=calibrate_model(g,d,method,cfg,0,1e-7)
                row=evaluate_model(g,d,r,method,0,1e-7); row.update(date=d["date"],kind="market")
                hits=[]
                for s,test in enumerate(d["tests"]):
                    p=(np.asarray(r["test_predictions"][s]) if "test_predictions" in r
                       else interpolation(g["K"],test["k"])@r["C"][s])
                    hits.extend((p>=test["bid"])&(p<=test["ask"]))
                row["heldout_bidask_fraction"]=float(np.mean(hits)); rows.append(row)
                save_json(out/"checkpoints"/(d["dataset_id"]+"_"+method+".json"),dict(row=row,result=r))
            except (ValueError,RuntimeError,FloatingPointError) as exc:
                failed.append(dict(dataset=d["dataset_id"],method=method,reason=str(exc)))
        pd.DataFrame(rows).to_csv(out/"market_results.csv",index=False)
    save_json(out/"market_status.json",dict(status="completed" if rows else "failed",dates=len(datasets),
              source_sha256=hashlib.sha256(Path(path).read_bytes()).hexdigest(),failures=failed))
    return pd.DataFrame(rows)


## 3. Actual small coherent circuit
Two coefficient registers, initial-state preparation, finite LCHS SELECT, off-grid quote dilation,
reference subtraction, and objective block-encoding. The code does not load a table of objective values.
Generic dense synthesis is expensive and is explicitly counted; this is not a scalable oracle.


In [ ]:
"""Explicit small circuits. Dense SELECT synthesis is costed, not called efficient.

No objective values are loaded into the circuit. The residual is constructed
from the LCHS propagator, the quote observation matrix and reference prices.
"""
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation, UnitaryGate
from qiskit.quantum_info import Statevector

def contraction_dilation(B):
    """Halmos unitary dilation; B is a contraction. Ancilla is the high bit."""
    U,s,Vh=sla.svd(B); s=np.clip(s,0,1)
    left=(U*np.sqrt(np.maximum(0,1-s*s)))@U.conj().T
    V=Vh.conj().T; right=(V*np.sqrt(np.maximum(0,1-s*s)))@V.conj().T
    full=np.block([[B,left],[right,-B.conj().T]])
    if not np.allclose(full.conj().T@full,np.eye(len(full)),atol=2e-10):
        raise AssertionError("Dilation is not unitary")
    return full

def select_condition(circuit,gate,controls,targets,value):
    """Little-endian integer convention, explicitly tested by statevector readback."""
    if not controls:
        circuit.append(gate,targets); return
    for j,bit in enumerate(controls):
        if not ((value>>j)&1): circuit.x(bit)
    circuit.append(gate.control(len(controls)),controls+targets)
    for j,bit in enumerate(controls):
        if not ((value>>j)&1): circuit.x(bit)

def reflection_success(width,ancillas):
    """2 P_success - I, with success defined by all listed ancillas being zero."""
    qc=QuantumCircuit(width,name="2P-I"); qc.global_phase=np.pi
    for a in ancillas: qc.x(a)
    if len(ancillas)==1: qc.z(ancillas[0])
    else: qc.mcp(np.pi,ancillas[:-1],ancillas[-1])
    for a in ancillas: qc.x(a)
    return qc

def build_residual_circuit(cfg):
    n=cfg["circuit_n"]; m=cfg["circuit_m"]; g=make_grid(n,2)
    N=g["N"]; d=2*m; D=2**d; J=cfg["circuit_terms"]; b=int(np.log2(J))
    if 2**b!=J: raise ValueError("Circuit LCU count must be a power of two")
    k,w,meta=lchs_weights("beta",J,4.); alpha=float(np.abs(w).sum())
    dt=float(TIMES[0]); c=g["cbar"]
    design=list(range(d)); work=list(range(d,d+n)); kr=list(range(d+n,d+n+b))
    wa=d+n+b; branch=wa+1; width=branch+1
    # Three distinct physical quote locations (two when N=4).
    qk=np.array([.85,1.1]) if N==4 else np.array([.8,1.,1.25])
    Wfull=interpolation(g["K"],qk); Wsmall=Wfull[:,1:-1]
    W=np.zeros((N,N)); W[:len(qk)]=Wsmall
    gammaW=max(1.,sla.norm(W,2)); UW=contraction_dilation(W/gammaW)
    xi_star=np.array([.7,.25]); exact=sla.expm(-dt*assemble(g,xi_star))@g["v0"]
    ref=Wsmall@exact; y=np.zeros(N); y[:len(ref)]=ref
    vnorm=sla.norm(g["v0"]); ynorm=sla.norm(y)
    a=vnorm*np.exp(c*dt)*alpha*gammaW; scale=a+ynorm
    prep=StatePreparation(np.sqrt(np.abs(w)/alpha))
    forward=QuantumCircuit(width-1,name="LCHS-W")
    forward.append(StatePreparation(g["v0"]/vnorm),work)
    forward.append(prep,kr)
    values=[]; losses=[]; forward_errors=[]
    for index in range(D):
        xi=np.array([index%(2**m),index//(2**m)])/(2**m-1)
        A=assemble(g,xi); L=(A+A.T)/2+c*np.eye(N); H=-.5j*(A-A.T)
        approx=np.zeros((N,N),complex)
        for j in range(J):
            U=sla.expm(-1j*dt*(H+k[j]*L))
            approx+=w[j]*U
            gate=UnitaryGate(np.exp(1j*np.angle(w[j]))*U,label="exp(-i(H+kL)t)")
            # controls ordered design then LCU index
            select_condition(forward,gate,design+kr,work,index+(j<<d))
        pv=np.exp(c*dt)*approx@g["v0"]
        loss=float(np.linalg.norm(W@pv-y)**2)
        values.append(xi); losses.append(loss)
        forward_errors.append(float(np.max(np.abs(pv-sla.expm(-dt*A)@g["v0"]))))
    forward.append(prep.inverse(),kr)
    forward.append(UnitaryGate(UW,label="quote W dilation"),work+[wa])
    V=QuantumCircuit(width,name="Residual")
    branchprep=StatePreparation([np.sqrt(a/scale),np.sqrt(ynorm/scale)])
    V.append(branchprep,[branch])
    V.x(branch); V.append(forward.to_gate().control(1),[branch]+list(range(width-1))); V.x(branch)
    V.append(StatePreparation(y/ynorm).control(1),[branch]+work)
    V.z(branch)
    V.append(branchprep.inverse(),[branch])
    return dict(V=V,g=g,d=d,D=D,N=N,values=np.array(values),losses=np.array(losses),
                scale=scale,ancillas=kr+[wa,branch],meta=meta,forward_errors=forward_errors,
                gammaW=gammaW,lcualpha=alpha,dt=dt)

def run_quantum_checks(cfg):
    out=cfg["out"]; print("Constructing explicit coherent residual circuit...")
    t=time.perf_counter(); built=build_residual_circuit(cfg)
    V=built["V"]; d=built["d"]; D=built["D"]; N=built["N"]; width=V.num_qubits
    initial=QuantumCircuit(width)
    for bit in range(d): initial.h(bit)
    residual_state=Statevector.from_instruction(initial.compose(V)).data.reshape(-1,D)
    probabilities=np.sum(np.abs(residual_state[:N])**2,axis=0)*D
    expected=built["losses"]/built["scale"]**2
    residual_error=float(np.max(np.abs(probabilities-expected)))
    assert residual_error<2e-8,"Residual circuit does not match matrix calculation"
    # V^dagger (2P-I) V is block-encoded by clean workspace. Add one
    # Hadamard-test flag so the clean block is exactly F / scale^2.
    R=reflection_success(width,built["ancillas"])
    U=QuantumCircuit(width,name="2F-1")
    U.compose(V,inplace=True); U.compose(R,inplace=True); U.compose(V.inverse(),inplace=True)
    F=QuantumCircuit(width+1,name="Objective BE")
    flag=width; F.h(flag); F.append(U.to_gate().control(1),[flag]+list(range(width))); F.h(flag)
    ini=QuantumCircuit(width+1)
    for bit in range(d): ini.h(bit)
    state=Statevector.from_instruction(ini.compose(F)).data
    observed=state[:D]*np.sqrt(D)
    oracle_error=float(np.max(np.abs(observed-expected)))
    assert oracle_error<3e-8,"Objective block-encoding mismatch"
    print("Transpiling dense small-instance oracle into u/cx gates...")
    compiled=transpile(F,basis_gates=["u","cx"],optimization_level=0,seed_transpiler=17)
    counts=compiled.count_ops()
    resource=dict(n=cfg["circuit_n"],M=2,m=cfg["circuit_m"],
                  design_points=D,lcu_terms=cfg["circuit_terms"],qubits=F.num_qubits,
                  depth=compiled.depth(),cx=int(counts.get("cx",0)),u=int(counts.get("u",0)),
                  residual_probability_error=residual_error,objective_block_error=oracle_error,
                  classical_build_validation_s=time.perf_counter()-t,
                  scale=built["scale"],alpha_lcu=built["lcualpha"],gamma_W=built["gammaW"],
                  cost_scope="Explicit dense design-enumerated SELECT; not asymptotic efficient synthesis",
                  gate_set="u,cx, all-to-all connectivity, no fault-tolerant compilation",hardware_execution=False)
    save_json(out/"quantum_resources.json",resource)
    frame=pd.DataFrame(dict(index=np.arange(D),xi1=built["values"][:,0],xi2=built["values"][:,1],
                  physical_loss=built["losses"],normalised_loss=expected,
                  circuit_probability=probabilities,circuit_block_real=observed.real,
                  circuit_block_imag=observed.imag,forward_price_error=built["forward_errors"]))
    frame.to_csv(out/"quantum_objective.csv",index=False)
    # Save a readable circuit diagram of the high-level validated composition.
    (out/"diagnostics"/"circuit.txt").write_text(str(F.draw(output="text",fold=100)))
    print("Quantum statevector checks PASS:",resource)
    return frame,resource

def run_qhd(cfg,landscape):
    """Classical simulation of QHD on the verified discrete objective landscape.
    The landscape is classically tabulated. This is not a full coherent run.
    """
    size=int(np.sqrt(len(landscape))); F=landscape.physical_loss.to_numpy()
    scale=max(float(np.ptp(F)),1e-12); V=(F-F.min())/scale
    h=1/(size-1)
    lap=np.diag(np.r_[1.,np.full(size-2,2.),1.])-np.diag(np.ones(size-1),1)-np.diag(np.ones(size-1),-1)
    kinetic=(np.kron(np.eye(size),lap)+np.kron(lap,np.eye(size)))/(2*h*h)
    eig,Q=sla.eigh(kinetic); psi=np.ones(size*size,complex)/size
    steps=cfg["qhd_steps"]; total=5.; dt=total/steps; records=[]
    target=F<=F.min()+.01*scale
    def kinetic_step(p,coef): return Q@(np.exp(-1j*coef*eig)*(Q.T@p))
    for j in range(steps+1):
        if j%max(1,steps//100)==0 or j==steps:
            p=np.abs(psi)**2
            records.append(dict(step=j,time=j*dt,expected_loss=float(p@F),
                                best_grid_loss=float(F.min()),near_optimal_probability=float(p@target),norm=float(p.sum())))
        if j==steps: break
        s=(j+.5)*dt
        psi=kinetic_step(psi,.5*dt*np.exp(-s))
        psi*=np.exp(-1j*dt*np.exp(s)*V)
        psi=kinetic_step(psi,.5*dt*np.exp(-s))
    assert abs(np.sum(np.abs(psi)**2)-1)<1e-8
    result=pd.DataFrame(records); result.to_csv(cfg["out"]/"qhd_trajectory.csv",index=False)
    final=landscape.copy(); final["probability"]=np.abs(psi)**2
    final.to_csv(cfg["out"]/"qhd_distribution.csv",index=False)
    save_json(cfg["out"]/"qhd_status.json",dict(scope="Classically tabulated landscape; split-step QHD simulation",
            strong_convexity_verified=False,global_continuous_optimum_verified=False,
            reference="Exact enumeration of this finite design grid",schedule="exp(-s) kinetic + exp(s) normalised potential"))
    return result


## 4. Additional sensitivities and figures
QHD below uses a classically tabulated denser landscape with a doubled-step check.
Its coarse corners are cross-checked against the circuit experiment; the dense grid is not a coherent experiment.


In [ ]:
"""Sensitivity analyses and QHD resolution checks; require preceding cells."""

def run_domain_slice_sensitivity(cfg):
    if not cfg["ablation"]:
        save_json(cfg["out"]/"sensitivity_status.json",dict(status="skipped_smoke_profile"))
        return pd.DataFrame()
    rows=[]; n=cfg["ns"][1]; gm=make_grid(n)
    for S in [2,4,8]:
        times=np.unique(np.r_[np.arange(1,S+1)/S,float(TIMES[0])])
        K,C=reference_surface("in",cfg["ref_n"],tuple(times))
        qs=[]; tests=[]; ks=TRAIN_K[0]; kt=np.linspace(.705,1.395,151)
        kt=kt[~np.any(np.isclose(kt[:,None],ks[None,:],atol=1e-10,rtol=0),axis=1)]
        for T,p in zip(times,C):
            mid=interpolation(K,ks)@p
            qs.append(dict(T=float(T),k=ks,mid=mid,bid=mid,ask=mid))
            tests.append(dict(T=float(T),k=kt,price=interpolation(K,kt)@p))
        data=dict(dataset_id=f"slice_{len(times)}",kind="synthetic",truth="in",noise_bp=0.,noise_seed=101,quotes=qs,tests=tests)
        for method in ["expm","lchs-beta"]:
            result=calibrate_model(gm,data,method,cfg,0,0.)
            row=evaluate_model(gm,data,result,method,0,0.)
            # The original two-slice coefficient-error indexing does not apply.
            row.pop("sigma2_linf",None)
            row.update(test="maturity_slices",nominal_slices=S,actual_slices=len(times),total_quotes=sum(len(q["k"]) for q in qs))
            rows.append(row)
            pd.DataFrame(rows).to_csv(cfg["out"]/"domain_slice_sensitivity.csv",index=False)
    data=make_dataset("in",0.,101,cfg["ref_n"])
    for domain in [1.,1.2,1.6]:
        # Fixed N means h changes: report it rather than calling this pure
        # boundary error. Independent fine-grid domain checks follow.
        g=make_grid(n,4,-domain,domain)
        r=calibrate_model(g,data,"expm",cfg,0,0.)
        row=evaluate_model(g,data,r,"expm",0,0.)
        row.update(test="fitted_domain",domain_halfwidth=domain,h=g["h"])
        rows.append(row)
    pts=np.linspace(.7,1.4,101)
    K0,C0=reference_surface("in",cfg["ref_n"]+1,tuple(TIMES),-1.6,1.6)
    for domain in [1.,1.2]:
        K,C=reference_surface("in",cfg["ref_n"],tuple(TIMES),-domain,domain)
        err=float(np.max(np.abs(C@interpolation(K,pts).T-C0@interpolation(K0,pts).T)))
        rows.append(dict(test="reference_domain",domain_halfwidth=domain,reference_domain_price_inf=err,
                         note="Includes finite reference discretisation error"))
    frame=pd.DataFrame(rows); frame.to_csv(cfg["out"]/"domain_slice_sensitivity.csv",index=False)
    save_json(cfg["out"]/"sensitivity_status.json",dict(status="completed"))
    return frame

def qhd_landscape_and_refinement(cfg,circuit_landscape):
    """Denser CLASSICAL tabulation of the same finite-LCHS objective.
    The actual circuit validates its coarse design points only. The denser
    landscape does not imply a coherent multi-bit experiment was performed.
    """
    size={"smoke":4,"pilot":8,"paper":16}[cfg["profile"]]
    g=make_grid(cfg["circuit_n"],2); dt=float(TIMES[0])
    qk=np.array([.85,1.1])
    W=interpolation(g["K"],qk)[:,1:-1]
    y=W@(sla.expm(-dt*assemble(g,[.7,.25]))@g["v0"])
    k,w,_=lchs_weights("beta",cfg["circuit_terms"],4.)
    rows=[]
    for j in range(size):
        for i in range(size):
            xi=np.array([i,j])/(size-1); A=assemble(g,xi)
            L=(A+A.T)/2+g["cbar"]*np.eye(g["N"]); H=-.5j*(A-A.T)
            p=sum(wl*(sla.expm(-1j*dt*(H+kl*L))@g["v0"]) for kl,wl in zip(k,w))*np.exp(g["cbar"]*dt)
            rows.append(dict(xi1=xi[0],xi2=xi[1],physical_loss=float(np.linalg.norm(W@p-y)**2)))
    landscape=pd.DataFrame(rows)
    for row in circuit_landscape.itertuples():
        match=landscape[(np.isclose(landscape.xi1,row.xi1))&(np.isclose(landscape.xi2,row.xi2))]
        assert len(match)==1 and abs(match.physical_loss.iloc[0]-row.physical_loss)<1e-12
    landscape.to_csv(cfg["out"]/"qhd_classical_landscape.csv",index=False)
    trajectory=run_qhd(cfg,landscape)
    fine=dict(cfg); fine["out"]=cfg["out"]/"diagnostics"/"qhd_refined"
    fine["out"].mkdir(parents=True,exist_ok=True); fine["qhd_steps"]=2*cfg["qhd_steps"]
    refined=run_qhd(fine,landscape)
    p=pd.read_csv(cfg["out"]/"qhd_distribution.csv").probability.to_numpy()
    q=pd.read_csv(fine["out"]/"qhd_distribution.csv").probability.to_numpy()
    row=dict(steps=cfg["qhd_steps"],refined_steps=fine["qhd_steps"],design_points=size*size,
             probability_total_variation=float(.5*np.abs(p-q).sum()),
             expected_loss_difference=float(abs(trajectory.expected_loss.iloc[-1]-refined.expected_loss.iloc[-1])),
             scope="Classical tabulation; coarse circuit points cross-checked; not full coherent QHD")
    pd.DataFrame([row]).to_csv(cfg["out"]/"qhd_refinement.csv",index=False)
    return trajectory


In [ ]:
"""Publication figures and evidence-only LaTeX tables; no simulated results supplied."""
import zipfile, shutil

COLORS={"expm":"#1764A0","fd":"#55813B","lchs-beta":"#D16A22","lchs-simple":"#8A619A",
        "AH":"#4B5961","PINN-fixed":"#918267","PINN-Wam":"#008D91"}
LABELS={"expm":"Exact expm","fd":"FD + adjoint","lchs-beta":"LCHS beta",
        "lchs-simple":"LCHS simple","AH":"Andreasen-Huge","PINN-fixed":"PINN fixed","PINN-Wam":"PINN adaptive"}

def figure_style():
    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":8,"axes.titlesize":8,
        "axes.labelsize":8,"xtick.labelsize":7,"ytick.labelsize":7,"legend.fontsize":6.5,
        "axes.spines.top":False,"axes.spines.right":False,"pdf.fonttype":42,
        "ps.fonttype":42,"savefig.dpi":220,"axes.axisbelow":True})

def export_figure(fig,cfg,name,sources,caption):
    for ax in fig.axes:
        if ax.get_label()!="<colorbar>": ax.grid(alpha=.16,linewidth=.5)
    if cfg["profile"]!="paper":
        fig.text(.99,-.075,cfg["profile"].upper()+" / implementation check",ha="right",fontsize=6,color="#80562D")
    fig.savefig(cfg["out"]/"figures"/(name+".pdf"),bbox_inches="tight",pad_inches=.06)
    fig.savefig(cfg["out"]/"figures"/(name+".png"),bbox_inches="tight",pad_inches=.06)
    save_json(cfg["out"]/"figures"/(name+".json"),dict(sources=sources,caption=caption,profile=cfg["profile"]))
    plt.close(fig)

def make_figures(cfg,results,selected):
    figure_style(); out=cfg["out"]; n=cfg["ns"][min(1,len(cfg["ns"])-1)]; g=make_grid(n)
    fig,ax=plt.subplots(2,2,figsize=(3.5,3.45),layout="constrained")
    for r,truth in enumerate(["in","out"]):
        data=make_dataset(truth,0.,101,cfg["ref_n"]); s=1
        plotk=np.linspace(.7,1.4,401)
        ax[r,0].plot(plotk,true_sigma2(np.log(plotk),s,truth),color="black",lw=1.3,label="Truth")
        for method in ["expm","lchs-beta"]:
            key=f'{data["dataset_id"]}_n{n}_{method}_s{cfg["seeds"][0]}'
            if key not in selected: continue
            result=selected[key]["result"]
            ax[r,0].plot(plotk,.01+np.asarray(result["hats"][s])@basis(np.log(plotk),g["M"]),color=COLORS[method],lw=1,label=LABELS[method])
            test=data["tests"][s]; p=interpolation(g["K"],test["k"])@np.asarray(result["C"])[s]
            ax[r,1].plot(test["k"],(p-test["price"])*1e4,color=COLORS[method],lw=1,label=LABELS[method]+" raw")
            if method=="lchs-beta":
                px=interpolation(g["K"],test["k"])@np.asarray(result["C_exact"])[s]
                ax[r,1].plot(test["k"],(px-test["price"])*1e4,color=COLORS[method],ls=":",lw=1,label="Beta exact re-eval.")
        ax[r,0].set(xlim=(.7,1.4),ylabel="Local variance",title=("In-family" if truth=="in" else "Out-of-family")+", T=1")
        ax[r,1].axhline(0,color=".6",lw=.6); ax[r,1].set(ylabel="Price error (bp)",title="Held-out strikes, T=1")
        for a in ax[r]: a.set_xlabel("Normalised strike k")
    ax[0,0].legend(loc="best"); ax[0,1].legend(loc="best")
    export_figure(fig,cfg,"calibration",["selected_examples.json","raw/"],"Seed 0, zero noise, second maturity; explicit dataset identifiers in selected_examples.json.")

    df=pd.read_csv(out/"error_budget.csv")
    fig,axs=plt.subplots(1,2,figsize=(3.5,2.15),layout="constrained")
    for kernel,color in [("simple",COLORS["lchs-simple"]),("beta",COLORS["lchs-beta"])]:
        part=df[(df.n==n)&(df.kernel==kernel)]
        for _,p in part.groupby("cutoff"):
            axs[0].loglog(p.terms,p.price_error_inf,":",color=color,alpha=.25,lw=.7)
        best=part.loc[part.groupby("terms").price_error_inf.idxmin()].sort_values("terms")
        axs[0].loglog(best.terms,best.price_error_inf,"o-",ms=3,color=color,label=kernel)
        axs[1].loglog(part.solver_s*1000,part.price_error_inf,"o",ms=3,color=color,label=kernel)
    axs[0].set(xlabel="LCU terms J",ylabel="Max price error",title=f"Cutoff envelope, N={2**n}")
    from matplotlib.ticker import NullFormatter
    axs[0].set_xticks(cfg["kernels"],[str(x) for x in cfg["kernels"]])
    axs[0].xaxis.set_minor_formatter(NullFormatter())
    timing=df[df.n==n].solver_s.to_numpy()*1000
    ticks=np.geomspace(timing.min(),timing.max(),3)
    axs[1].set_xticks(ticks,[f"{x:.2g}" for x in ticks]); axs[1].xaxis.set_minor_formatter(NullFormatter())
    axs[1].set(xlabel="Warm evaluation (ms)",ylabel="Max price error",title="Classical simulation")
    axs[0].legend(); axs[1].legend()
    export_figure(fig,cfg,"kernel_budget",["error_budget.csv"],"Faint lines show every cutoff; envelope minimises reference error and is not a deployable tuning rule.")

    fig,axs=plt.subplots(1,3,figsize=(3.5,1.85),layout="constrained")
    for ax,(kind,label) in zip(axs,[("butterfly","Butterfly"),("calendar","Calendar"),("density","Density")]):
        x=np.maximum(df[kind+"_bound"].to_numpy(),1e-16); y=np.maximum(df[kind+"_violation"].to_numpy(),1e-16)
        ax.loglog(x,y,"o",ms=2.3,color=COLORS["lchs-beta"],alpha=.6)
        low=min(x.min(),y.min()); high=max(x.max(),y.max()); ax.plot([low,high],[low,high],"--",color=".5",lw=.7)
        ax.set(title=label,xlabel="Bound",ylabel="Violation" if kind=="butterfly" else "")
    export_figure(fig,cfg,"error_certificate",["error_budget.csv"],"Zero violations plotted at 1e-16. Units are price for butterfly/calendar, price per strike squared for density. All bounds are conditional on measured exact-model price error.")

    fig,ax=plt.subplots(figsize=(5.5,3.2),layout="constrained")
    part=results[(results.n==n)&(results.truth=="out")&(results.noise_bp==5)]
    methods=[m for m in LABELS if m in set(part.method)]
    for i,m in enumerate(methods):
        values=part[part.method==m].rmse_heldout.to_numpy()*1e4
        ax.scatter(values,i+np.linspace(-.1,.1,len(values)),color=COLORS[m],s=13,alpha=.7)
    ax.set_yticks(range(len(methods)),[LABELS[m] for m in methods]); ax.invert_yaxis()
    ax.set(xlabel="Held-out price RMSE (normalised bp)",title=f"Out-of-family, 5 bp noise, N={2**n}")
    export_figure(fig,cfg,"benchmark_variability",["results.csv"],"Each dot is an executed run; initialisation seeds are not independent market observations.")

    q=pd.read_csv(out/"quantisation.csv"); f=pd.read_csv(out/"fourier_basis.csv")
    fig,axs=plt.subplots(1,2,figsize=(5.5,2.4),layout="constrained")
    axs[0].semilogy(q.bits,q.rmse_quotes,"o-",color=COLORS["expm"],label="Quote RMSE")
    axs[0].semilogy(q.bits,q.rmse_heldout,"s-",color=COLORS["lchs-beta"],label="Held-out RMSE")
    axs[0].axhline(q.continuous_reference.iloc[0],ls=":",color=".4",label="Continuous quote RMSE")
    axs[0].set(xlabel="Bits per coefficient",ylabel="Price RMSE",title="Rounded fitted coefficients"); axs[0].legend()
    axs[1].loglog(f.degree,f.error_inf,"o-",color=COLORS["expm"],label="Basis sup error")
    axs[1].loglog(f.degree,np.maximum(-f.minimum_basis,1e-18),"s--",color=COLORS["lchs-beta"],label="Negative basis overshoot")
    axs[1].set(xlabel="Fourier degree",ylabel="Absolute basis error",title="Compact C1 basis"); axs[1].legend()
    export_figure(fig,cfg,"precision_basis",["quantisation.csv","fourier_basis.csv"],"Quantisation does not re-optimise the discrete objective. Fourier approximation is diagnosed separately from exact pointwise calibration.")

    q=pd.read_csv(out/"quantum_objective.csv")
    fig,axs=plt.subplots(1,2,figsize=(3.5,1.85),layout="constrained")
    axs[0].plot(q.normalised_loss,q.circuit_block_real,"o",ms=4,color=COLORS["expm"])
    lo=q.normalised_loss.min(); hi=q.normalised_loss.max(); axs[0].plot([lo,hi],[lo,hi],"--",color=".5",lw=.7)
    axs[0].set(xlabel="Matrix objective / scale²",ylabel="Circuit clean block",title="All coherent designs")
    err=np.abs(q.circuit_block_real+1j*q.circuit_block_imag-q.normalised_loss)
    axs[1].semilogy(q.index,np.maximum(err,1e-16),"o-",color=COLORS["lchs-beta"],ms=3)
    axs[1].set(xlabel="Design index",ylabel="Block error",title="Statevector validation")
    export_figure(fig,cfg,"quantum_validation",["quantum_objective.csv","quantum_resources.json"],"Actual residual/objective circuit, one maturity and two design parameters; dense enumerated SELECT.")

    q=pd.read_csv(out/"qhd_trajectory.csv"); d=pd.read_csv(out/"qhd_distribution.csv")
    fig,axs=plt.subplots(1,2,figsize=(5.5,2.4),layout="constrained")
    axs[0].plot(q.time,q.near_optimal_probability,color=COLORS["expm"])
    axs[0].axhline(q.near_optimal_probability.iloc[0],ls=":",color=".5",label="Uniform initial state")
    axs[0].set(xlabel="Schedule time",ylabel="Near-grid-optimum probability",ylim=(0,1),title="Classical QHD simulation"); axs[0].legend()
    sz=int(np.sqrt(len(d))); im=axs[1].imshow(d.probability.to_numpy().reshape(sz,sz),origin="lower",extent=[0,1,0,1],aspect="auto",cmap="Blues")
    axs[1].set(xlabel="Coefficient 1",ylabel="Coefficient 2",title="Final design probability"); fig.colorbar(im,ax=axs[1],label="Probability")
    export_figure(fig,cfg,"qhd_diagnostics",["qhd_trajectory.csv","qhd_distribution.csv","qhd_refinement.csv"],"Classically tabulated denser landscape; only the separate coarse design circuit is coherently verified. Near optimal = within 1% of the finite objective range.")

    f=pd.read_csv(out/"fd_convergence.csv"); j=pd.read_csv(out/"identifiability.csv")
    fig,axs=plt.subplots(1,2,figsize=(5.5,2.3),layout="constrained")
    axs[0].loglog(f.steps,f.price_error_inf,"o-",color=COLORS["fd"]); axs[0].set(xlabel="Backward Euler steps",ylabel="Max price error",title="Time discretisation at fixed design")
    for s,p in j.groupby("slice"): axs[1].semilogy(p["index"],p.singular_value,"o-",label=f"Slice {s}")
    axs[1].set(xlabel="Singular-value index",ylabel="Quote-Jacobian singular value",title="Local identifiability"); axs[1].legend()
    export_figure(fig,cfg,"fd_identifiability",["fd_convergence.csv","identifiability.csv"],"FD convergence separates time error from optimiser error. Jacobian spectra are local diagnostics.")

    fig,axs=plt.subplots(1,2,figsize=(5.5,2.3),layout="constrained")
    for method in ["PINN-fixed","PINN-Wam"]:
        key=f'out_bp5_d101_n{n}_{method}_s{cfg["seeds"][0]}'
        if key not in selected: continue
        hist=pd.DataFrame(selected[key]["result"]["pinn_history"])
        for ax,metric in zip(axs,["fit","pde"]): ax.semilogy(hist.step,np.maximum(hist[metric],1e-18),label=LABELS[method],color=COLORS[method])
    for ax,metric in zip(axs,["Unweighted fit MSE","Unweighted PDE MSE"]):
        ax.set(xlabel="Training step",ylabel=metric,title="Out-of-family / seed 0"); ax.legend()
    export_figure(fig,cfg,"pinn_losses",["selected_examples.json"],"Fixed/adaptive network losses; adaptation is not an exact reproduction of the cited study.")

    if (out/"ablation.csv").exists():
        ab=pd.read_csv(out/"ablation.csv"); fig,axs=plt.subplots(1,2,figsize=(5.5,2.3),layout="constrained")
        for ax,method in zip(axs,["expm","lchs-beta"]):
            p=ab[ab.method==method].pivot(index="M",columns="lam",values="rmse_heldout")
            im=ax.imshow(p.to_numpy()*1e4,aspect="auto",cmap="Blues",vmin=ab.rmse_heldout.min()*1e4,vmax=ab.rmse_heldout.max()*1e4)
            ax.set_xticks(range(len(p.columns)),[f"{x:g}" for x in p.columns],rotation=30)
            ax.set_yticks(range(len(p.index)),p.index); ax.set(xlabel="Regularisation lambda",ylabel="Basis size M",title=LABELS[method]); fig.colorbar(im,ax=ax,label="Held-out RMSE (bp)")
        export_figure(fig,cfg,"ablation",["ablation.csv"],"Out-of-family, fixed 5 bp noise dataset and initialisation. Not a hyperparameter selection study.")
    if (out/"domain_slice_sensitivity.csv").exists():
        p=pd.read_csv(out/"domain_slice_sensitivity.csv"); fig,axs=plt.subplots(1,2,figsize=(5.5,2.3),layout="constrained")
        for method,d in p[p.test=="maturity_slices"].groupby("method"):
            axs[0].plot(d.actual_slices,d.rmse_heldout*1e4,"o-",label=LABELS[method],color=COLORS[method])
        d=p[p.test=="fitted_domain"]; axs[1].plot(d.domain_halfwidth,d.rmse_heldout*1e4,"o-",color=COLORS["expm"])
        axs[0].set(xlabel="Actual maturity slices",ylabel="Held-out RMSE (bp)",title="Extra expiries add observations"); axs[0].legend()
        axs[1].set(xlabel="Log-strike domain half-width",ylabel="Held-out RMSE (bp)",title="Fixed N: boundary and grid effects")
        export_figure(fig,cfg,"domain_slices",["domain_slice_sensitivity.csv"],"The true 182/365 breakpoint is retained. More expiries add data; fixed N domain changes also alter h.")

def tex_escape(s):
    return str(s).replace("\\",r"\textbackslash{}").replace("_",r"\_").replace("%",r"\%").replace("&",r"\&").replace("#",r"\#")

def tex_number(x):
    if not np.isfinite(x): return "--"
    return f"{x:.3g}"

def write_tables(cfg,results):
    out=cfg["out"]; t=out/"tables"; n=cfg["ns"][min(1,len(cfg["ns"])-1)]
    failures=json.loads((out/"failures.json").read_text())
    status=json.loads((out/"market_status.json").read_text())
    market_status=status["status"]
    macro=("\\renewcommand{\\StudyProfile}{"+cfg["profile"]+"}\n"+
           "\\renewcommand{\\RunCount}{"+str(len(results))+"}\n"+
           "\\renewcommand{\\MarketStatus}{"+tex_escape(market_status)+"}\n"+
           "\\renewcommand{\\RunStatus}{"+str(len(failures))+" failed runs are retained in the failure log.}\n")
    (t/"run_macros.tex").write_text(macro)
    # Dataset means prevent initialisation replication from becoming fake data n.
    part=results[(results.n==n)&(results.truth=="out")&(results.noise_bp==5)]
    cols=["rmse_quotes","rmse_heldout","iv_rmse","opt_s","butterfly_violation","calendar_violation"]
    means=part.groupby(["method","dataset"],as_index=False)[cols].mean()
    abstract=""
    if cfg["profile"]=="paper":
        vals={m:float(d.rmse_heldout.mean()*1e4) for m,d in means.groupby("method")}
        if all(m in vals for m in ["expm","fd","lchs-beta","AH"]):
            abstract=(f"For the out-of-family, 5-bp-noise case at $N={2**n}$, mean held-out price errors are "
                f"{vals['lchs-beta']:.3g}, {vals['expm']:.3g}, {vals['fd']:.3g}, and {vals['AH']:.3g} normalised bp "
                "for beta-LCHS, exact propagation, finite differences, and Andreasen--Huge, respectively.")
    with (t/"run_macros.tex").open("a") as handle:
        handle.write("\\renewcommand{\\AbstractEvidence}{"+abstract+"}\n")
    means.to_csv(out/"dataset_means.csv",index=False)
    lines=[r"\begin{table}[t]\centering\caption{Out-of-family, 5 bp noise, $N="+str(2**n)+r"$. Means over datasets after averaging initialisations. Price errors are normalised bp; time is classical optimisation seconds.}",
           r"\label{tab:bench}\scriptsize\begin{tabular}{lrrrr}\toprule Method & Fit & Test & Seconds & Runs\\\midrule"]
    for method in LABELS:
        d=means[means.method==method]
        if d.empty: continue
        lines.append(tex_escape(LABELS[method])+" & "+" & ".join([tex_number(d.rmse_quotes.mean()*1e4),tex_number(d.rmse_heldout.mean()*1e4),tex_number(d.opt_s.mean()),str(int((part.method==method).sum()))])+r"\\")
    lines += [r"\bottomrule\end{tabular}\end{table}"]
    (t/"benchmark.tex").write_text("\n".join(lines))
    # Full uncertainty summaries retain both hierarchy levels in CSV.
    metrics=[c for c in cols+["sigma2_linf","compile_warm_s","iv_valid","iv_total"] if c in results]
    groups=["truth","noise_bp","n","method"]
    results.groupby(groups)[metrics].agg(["mean","std","min","max","count"]).to_csv(out/"all_run_summary.csv")
    dataset=results.groupby(groups+["dataset"])[metrics].mean()
    dataset.groupby(groups)[metrics].agg(["mean","std","count"]).to_csv(out/"between_dataset_summary.csv")
    results.groupby(groups+["dataset"])[metrics].std().to_csv(out/"within_dataset_initialisation_sd.csv")
    lc=results[results.method.str.startswith("lchs")]
    exact=results[results.method=="expm"]
    summary=(f"The executed {tex_escape(cfg['profile'])} profile contains {len(results)} completed synthetic calibration runs on "
       f"{results.dataset.nunique()} distinct datasets; {len(failures)} failures are logged. "
       f"Raw LCHS price-butterfly and calendar violation maxima are {lc.butterfly_violation.max():.3g} and {lc.calendar_violation.max():.3g}, respectively. "
       f"The exact-exponential price-butterfly violation maximum is {exact.butterfly_violation.max():.3g}. "
       "These are normalised price quantities, not derivative tolerances. The complete per-run CSV retains non-convergence, invalid-IV counts, compilation costs, and exact re-evaluation audits. ")
    if cfg["profile"]!="paper": summary+="These reduced-budget results are implementation evidence only and must not be presented as the full study. "
    summary+="Between-dataset standard deviations and within-dataset initialisation standard deviations are exported separately; a single dataset does not support a between-dataset uncertainty estimate.\n"
    (t/"results_summary.tex").write_text(summary)
    resource=json.loads((out/"quantum_resources.json").read_text())
    text=(f"The executed one-slice circuit uses {resource['qubits']} qubits and {resource['design_points']} coherent design points. "
          f"The maximum clean-block error is {resource['objective_block_error']:.3g}. "
          f"Dense synthesis produces {resource['cx']:,} CNOT gates, {resource['u']:,} single-qubit gates, and depth {resource['depth']:,}. "
          "These counts use the declared all-to-all u/CNOT basis and generic design enumeration; they are not a fault-tolerant resource estimate.\n")
    (t/"quantum_summary.tex").write_text(text)
    if market_status=="not_run":
        mt="Real-market study: not run. No market observations were supplied. The input template and validation/held-out evaluation code are provided; no synthetic result is labelled as market evidence.\n"
    else:
        market=pd.read_csv(out/"market_results.csv")
        mt=(f"The market input contains {status.get('dates',0)} date/underlying datasets with {len(market)} completed method runs and {len(status.get('failures',[]))} failures. "
            "The source checksum, raw normalised snapshots, held-out bid--ask coverage and full method results accompany the manuscript. ")
        if len(market):
            mt+=r"\begin{table}[t]\centering\caption{Executed real-market runs: held-out price error and bid--ask coverage.}\scriptsize\begin{tabular}{lrr}\toprule Method & RMSE (bp) & Coverage\\\midrule"+"\n"
            for m,d in market.groupby("method"):
                mt+=tex_escape(LABELS[m])+f" & {1e4*d.rmse_heldout.mean():.3g} & {d.heldout_bidask_fraction.mean():.3f}"+r"\\"+"\n"
            mt+=r"\bottomrule\end{tabular}\end{table}"
    (t/"market_summary.tex").write_text(mt)
    save_json(out/"evidence_status.json",dict(profile=cfg["profile"],completed_runs=len(results),failures=len(failures),
             market_status=market_status,quantum_hardware_executed=False,full_coherent_qhd_executed=False,
             evidence_ready_for_author_review=cfg["profile"]=="paper" and market_status=="completed"))

def make_supplement(cfg):
    out=cfg["out"]
    intro=r"""\documentclass[10pt]{article}
\usepackage[margin=20mm]{geometry}
\usepackage{graphicx,booktabs,amsmath}
\usepackage[hidelinks]{hyperref}
\title{Supplementary diagnostics: local-volatility calibration}
\author{Anonymous submission}
\date{}
\begin{document}\maketitle
This supplement is generated from executed experiments. Profile: PROFILE.
Smoke and pilot outputs are validation/exploration, not final study evidence.
Every figure has a same-name JSON source record. Full metrics, unsuccessful
optimisation runs, failed baseline calls, and invalid implied-volatility counts
are retained in the accompanying CSV/JSON files.
""".replace("PROFILE",cfg["profile"])
    names=[("benchmark_variability","All executed initialisations, out-of-family noisy case."),
           ("precision_basis","Coefficient rounding and global Fourier approximation."),
           ("fd_identifiability","Time-discretisation and local identifiability diagnostics."),
           ("pinn_losses","Documented fixed/adaptive PINN comparison."),
           ("qhd_diagnostics",r"Classically tabulated QHD simulation; see qhd\_refinement.csv."),
           ("ablation","Independent basis-size and regularisation ablation."),
           ("domain_slices","Maturity and finite-domain sensitivity, with explicit confounders.")]
    for name,caption in names:
        if (out/"figures"/(name+".pdf")).exists():
            intro+=r"\begin{figure}[htbp]\centering\includegraphics[width=.92\linewidth]{figures/"+name+r".pdf}\caption{"+caption+r"}\end{figure}"+"\n"
    intro+=r"\end{document}"
    (out/"supplement.tex").write_text(intro)

def finalise_outputs(cfg,results,selected,main_tex,readme,source_files):
    make_figures(cfg,results,selected); write_tables(cfg,results); make_supplement(cfg)
    out=cfg["out"]; (out/"main.tex").write_text(main_tex,encoding="utf-8")
    (out/"README.md").write_text(readme,encoding="utf-8")
    (out/"source").mkdir(exist_ok=True)
    for name,content in source_files.items(): (out/"source"/name).write_text(content,encoding="utf-8")
    manifest=json.loads((out/"manifest.json").read_text()); manifest["study_status"]="completed"
    manifest["market_status"]=json.loads((out/"market_status.json").read_text())["status"]
    manifest["source_sha256"]={name:hashlib.sha256(content.encode()).hexdigest() for name,content in source_files.items()}
    manifest["manuscript_sha256"]=hashlib.sha256(main_tex.encode()).hexdigest()
    save_json(out/"manifest.json",manifest)
    archive=out.parent/(out.name+"_results.zip")
    with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.rglob("*")):
            if p.is_file() and p.suffix not in [".zip",".tmp"]: z.write(p,p.relative_to(out))
    print("Generated",len(list((out/"figures").glob("*.pdf"))),"figure PDFs, LaTeX tables, manuscript and supplement.")
    print("Result archive:",archive)
    return archive


## 5. Initialise and validate
The fixed fine reference is generated once, independently of each fitted grid. Reference construction
time is saved separately. Failed structural/gradient assertions stop execution.


In [ ]:
cfg=configure(PROFILE,OUTPUT_ROOT,MARKET_CSV)
reference_t0=time.perf_counter()
for truth in ["in","out"]:
    reference_surface(truth,cfg["ref_n"])
save_json(cfg["out"]/"reference_timing.json",dict(reference_generation_s=time.perf_counter()-reference_t0,
          reference_n=cfg["ref_n"],scope="Primary synthetic truth, before calibration; refinement timed separately by validation cell"))
validation_t0=time.perf_counter()
validation=run_validation(cfg)
save_json(cfg["out"]/"validation_timing.json",dict(validation_and_refinement_s=time.perf_counter()-validation_t0))
display(validation)


## 6. Matched solver and broader baseline comparisons
This is usually the longest section. Completed calibration runs are checkpointed.
Reported `opt_s` excludes first-call compile/warm-up and reference data generation.
Dense classical simulation of an LCHS sum is not a quantum execution time.


In [ ]:
results,selected=run_calibration_suite(cfg)
display(results.groupby(["truth","noise_bp","n","method"])[["rmse_quotes","rmse_heldout","opt_s"]].mean())
print("Failed calls:",json.loads((cfg["out"]/"failures.json").read_text()))


## 7. Solver-error budgets and raw no-arbitrage bounds
Both kernels are tested at every declared term budget and cutoff. A normalisation defect and LCU norm
are retained. The error certificates are conditional on measured exact-model price error.


In [ ]:
budget=run_error_budget(cfg)
display(budget[["n","kernel","terms","cutoff","price_error_inf","certificate_pass","solver_s"]])


## 8. Quantisation, basis, regularisation, identifiability, maturities and domain
The 4bit case is explicitly tested. Rounding is not re-optimisation on the discrete grid.
The compact C1 basis is not treated as a globally finite low-degree Fourier series.


In [ ]:
quantisation=run_diagnostics(cfg,selected)
sensitivity=run_domain_slice_sensitivity(cfg)
display(quantisation[["bits","rmse_quotes","rmse_heldout","design_qubits","inverse_hxi_squared"]])


## 9. Circuit validation and resource counts
The default circuit is deliberately small (9 total qubits). Dense synthesis can still be large.
Both the residual probability and clean objective block must match the matrix calculation.


In [ ]:
landscape,resources=run_quantum_checks(cfg)
display(landscape)
display(pd.DataFrame([resources]))


## 10. Separate classical QHD simulation
The potential is the finite LCHS residual objective. This step tabulates it classically.
Finite-grid enumeration supplies a reference; strong convexity and a continuum optimum are not assumed.
Inspect time-step sensitivity before interpreting optimisation behaviour.


In [ ]:
qhd=qhd_landscape_and_refinement(cfg,landscape)
display(pd.read_csv(cfg["out"]/"qhd_refinement.csv"))


## 11. Real-market CSV (optional input, explicitly tracked)
To run market experiments, provide a CSV path in the configuration cell before running the notebook.
The output includes an empty header template if no path is supplied. No synthetic observations are
used as a replacement for market evidence. Required schema and exclusions are in the embedded README.


In [ ]:
market=run_market(cfg)
display(market if len(market) else pd.DataFrame(columns=MARKET_COLUMNS))


## 12. Embedded source archive
The embedded sources make this notebook self-contained. They are copied into the result ZIP, with execution
tables and figures generated in the next cell. No external original notebook is required. The manuscript
draft that the original run filled with its results is not distributed; `main.tex` is a placeholder.


In [ ]:
MAIN_TEX = '% Placeholder. The manuscript draft that the original run of this notebook filled with its results\n% is not distributed with the repository.\n'
README_TEXT = '# Experiment notebook of the archived benchmark\n\nThis notebook, `optional/full_legacy_experiment.ipynb` in the repository and `local_volatility_revision.ipynb` at the time of the archived run, runs on its own. No external Python source and no other notebook are needed.\n\n1. In Google Colab, open the IPYNB with "File → Upload notebook".\n2. Set `PROFILE` in the first configuration cell. The default, `paper`, runs all comparisons and sensitivity analyses. Change it to `smoke` to check the connection first, or to `pilot` for a preliminary study.\n3. Choose "Runtime → Run all". At the end, a link appears to the result ZIP with the figures (PDF and PNG), the tables, the CSV files and the TeX of the diagnostics supplement.\n\n## Run settings\n\n| Setting | smoke | pilot | paper |\n|---|---:|---:|---:|\n| Interior points in space | 8, 16 | 32, 64 | 32, 64, 128 |\n| Interior points of the independent reference solution | 128 | 512 | 2,048 |\n| Initialisation seeds | 1 | 2 | 5 |\n| Seeds of the noisy data | 1 | 1 | 3 |\n| LCHS terms in the calibration | 64 | 512 | 2,048 |\n| PINN training steps | 30 | 1,500 | 6,000 |\n| Further comparisons of basis, regularisation, boundary and maturities | omitted | run | run |\n\nThe full `paper` experiment may take days on a CPU. Check with `smoke` or `pilot` first, and save checkpoints to Drive for the full experiment. On the CPU used when this was prepared, one warm evaluation of the LCHS objective and its gradient at N=128 with 2,048 terms took about 39 seconds. This is a check of a single setting and does not guarantee the duration of the full experiment.\n\n`paper` includes many eigendecompositions, neural-network training and comparison experiments, and runs for a long time. The run time depends on the runtime. It also works on a CPU. To use a GPU, first select a GPU runtime in Colab and set `USE_GPU=True` in the installation cell. Because 64-bit arithmetic is used, the speed-up from a GPU depends on the model. The GPU path was not part of the CPU verification of the archived run.\n\nTo resume a long experiment, set `MOUNT_DRIVE=True`; after authentication, the results are saved in the specified folder. Calibration checkpoints are reused for runs whose settings and input-file checksums match. A run that stopped part-way can resume from the last completed calibration. The individual diagnostic cells are run again. Failed calibrations are also recorded and reused, so delete the corresponding `*_failure.json` to retry one deliberately. Use a separate output folder to compare with new settings.\n\n## Real-market data\n\nNo market data are bundled. With an empty `MARKET_CSV` the market experiment is reported as "not run" and the other experiments are run. A `market_template.csv` with the header only is provided. At least two maturities with at least six rows each are needed. The input columns are as follows.\n\n| Column | Content |\n|---|---|\n| date, expiry | YYYY-MM-DD, valuation date and expiry date. ACT/365F is used |\n| underlying | identifier of the underlying |\n| strike | strike in the original currency |\n| forward, discount | forward and discount factor of the maturity. They agree within a maturity |\n| bid, ask | option prices at the same time, in units of the original currency |\n| option_type | `call` or `put`. Puts are converted by put-call parity |\n| exercise_style | `european` |\n| carry_model | `deterministic_proportional` |\n| source | data provider and snapshot identifier. Must not be empty |\n\nDo not enter both the call and the put for the same underlying, date, maturity and strike. Duplicates, expired contracts, crossed spreads, non-numeric values, unsupported exercise or carry, and strikes outside the computational domain are not removed silently; they raise a validation error. Do not treat discrete cash dividends of single stocks or American prices as European data.\n\n## Interpretation\n\nThe results of `smoke` serve to verify the program. The figures carry an explicit label. Numbers from a small-scale test must not be read as results of the full experiment.\n\nThe quantum circuit is fixed to a small verification with two coefficients, one bit each, and one maturity. The finite-LCHS objective is actually assembled, and objective values are not loaded into the circuit directly. SELECT, however, is a design enumeration and a general decomposition of a dense unitary, not an efficient oracle implementation. The gate counts become large. The QHD of `paper` uses the same finite-LCHS objective, tabulated classically on a 16×16 grid. What the quantum circuit verifies is the 2×2 set of design points. No fully coherent calibration, execution on quantum hardware, quantum advantage or global optimality of the continuous problem is claimed.\n\nThe PINN comparison is an **adapted version** that refers to the [public source](https://github.com/khoshisashi/DC-PINNs/blob/0cc9093e1ac9ebfdf45b81edadd97d6c80ae0157/DC_PINNs_IVS_calibration.ipynb). It implements independent IV and local-variance networks with tanh MLPs, fixed collocation, Adam, gradient balancing every 100 steps and an increase of the point weights with an offset of 50 steps. A denominator epsilon and a weight cap were added, and the PDE residual was kept from becoming a self-identity. It is not described as a reproduction of the numbers of the original paper.\n\nThe four PDE methods with the same basis are the main object of the solver comparison. AH and PINN are calibration baselines that differ in expressive power and objective, and their own quote and test predictions are evaluated. The time comparison keeps that difference and the compilation and warm-up. No significance test is made that treats initialisation seeds as independent market samples.\n\nIn the experiment with additional maturities, the true coefficient change point at 182/365 years is kept, so the actual number of slices is 3, 5 and 9 for the nominal 2, 4 and 8 divisions. Additional maturities also increase the number of observations. In the boundary experiment h changes as well at fixed N, so it is not called a pure boundary error. Quantisation is a rounding of the existing continuous solution, not a re-optimisation over the discrete design.\n\n## Outputs and traceability\n\n- `results.csv`: all synthetic runs. Raw and exact audits, IV valid/total, times, convergence flags.\n- `failures.json`: failed comparisons are kept as well.\n- `all_run_summary.csv`: descriptive statistics of all runs.\n- `between_dataset_summary.csv` / `within_dataset_initialisation_sd.csv`: the variation between data sets and between initialisations, separated.\n- `error_budget.csv`: all J/cutoff settings of both kernels, raw errors, normalisation coefficients, conditional bounds.\n- `quantum_objective.csv` / `quantum_resources.json`: agreement of the coherent circuit and gate counts.\n- `qhd_refinement.csv`: total variation of the probability distribution under a halving of the time step. If it is not small, increase the number of steps and run again.\n- `manifest.json`: settings, Python and package versions, execution device, source hashes, execution status.\n- `figures/*.pdf` / `*.png` / `*.json`: the figures and the correspondence with their source data.\n- `supplement.tex` / `tables/*.tex`: the diagnostics supplement and the tables generated from the results.\n- `main.tex`: a placeholder. The manuscript draft of the original run is not distributed.\n\nThe dependencies are JAX, SciPy, NumPy, pandas, Matplotlib, Qiskit and QuantLib. Install JAX for a GPU by the [official installation instructions](https://docs.jax.dev/en/latest/installation.html).\n'
SOURCE_FILES = {'core.py': '"""Reproducible local-volatility calibration; embedded verbatim in the notebook.\n\nAll prices are forward-normalised. No market observations are bundled.\n"""\nfrom pathlib import Path\nfrom functools import lru_cache, partial\nimport os, sys, json, time, hashlib, platform, importlib.metadata, warnings\nimport numpy as np\nimport pandas as pd\nimport scipy.linalg as sla\nimport scipy.sparse as sp\nimport scipy.sparse.linalg as spla\nfrom scipy.optimize import minimize, brentq\nfrom scipy.special import ndtr\nimport jax\njax.config.update("jax_enable_x64", True)\nimport jax.numpy as jnp\nfrom jax.flatten_util import ravel_pytree\nimport matplotlib\nimport matplotlib.pyplot as plt\n\nVERSION = "2026-09-05-r2"\nTIMES = np.array([182.0 / 365.0, 1.0])\nTRAIN_K = [np.array([.7,.85,.95,1.,1.05,1.2,1.4]),\n           np.array([.75,.9,1.,1.1,1.3])]\nXI_TRUTH = np.array([[.8,.3,.25,.7],[.6,.35,.3,.5]])\nPROFILES = {\n    "smoke": dict(ns=[3,4], ref_n=7, seeds=[0], noise_seeds=[101],\n                  kernels=[16,32,64], lchs_terms=64, maxiter=12, fd_steps=24,\n                  pinn_steps=30, pinn_width=12, collocation=48, ablation=False,\n                  circuit_n=2, circuit_m=1, circuit_terms=4, qhd_steps=80),\n    "pilot": dict(ns=[5,6], ref_n=9, seeds=[0,1], noise_seeds=[101],\n                  kernels=[128,256,512,1024], lchs_terms=512, maxiter=120,\n                  fd_steps=256, pinn_steps=1500, pinn_width=24, collocation=256,\n                  ablation=True, circuit_n=2, circuit_m=1, circuit_terms=4,\n                  qhd_steps=600),\n    "paper": dict(ns=[5,6,7], ref_n=11, seeds=list(range(5)),\n                  noise_seeds=[101,202,303], kernels=[256,512,1024,2048,4096],\n                  lchs_terms=2048, maxiter=220, fd_steps=1024,\n                  pinn_steps=6000, pinn_width=32, collocation=768,\n                  ablation=True, circuit_n=2, circuit_m=1, circuit_terms=4,\n                  qhd_steps=1500),\n}\n\ndef jsonable(v):\n    if isinstance(v, dict): return {str(k):jsonable(x) for k,x in v.items()}\n    if isinstance(v, (list,tuple)): return [jsonable(x) for x in v]\n    if isinstance(v, (np.ndarray,jax.Array)): return jsonable(np.asarray(v).tolist())\n    if isinstance(v, np.generic): return v.item()\n    if isinstance(v, Path): return str(v)\n    return v\n\ndef save_json(path, obj):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+".tmp")\n    tmp.write_text(json.dumps(jsonable(obj),indent=2,allow_nan=True),encoding="utf-8")\n    tmp.replace(path)\n\ndef configure(profile="paper", output_root="cifer_outputs", market_csv=""):\n    if profile not in PROFILES: raise ValueError("profile must be smoke, pilot or paper")\n    cfg=dict(PROFILES[profile], profile=profile, version=VERSION, market_csv=market_csv)\n    if market_csv:\n        cfg["market_input_sha256"]=hashlib.sha256(Path(market_csv).read_bytes()).hexdigest()\n    digest=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:12]\n    out=Path(output_root)/f"{profile}_{digest}"\n    for name in ["figures","tables","checkpoints","raw","diagnostics"]:\n        (out/name).mkdir(parents=True,exist_ok=True)\n    cfg["out"]=out\n    cfg["fingerprint"]=digest\n    cfg["backend"]=str(jax.devices())\n    versions={m:importlib.metadata.version(m) for m in\n              ["numpy","scipy","pandas","matplotlib","jax","qiskit","QuantLib"]}\n    save_json(out/"manifest.json",dict(config=cfg,packages=versions,\n              python=sys.version,platform=platform.platform(),\n              study_status="not_completed",market_status="not_run",\n              time_units="ACT/365F years",price_units="C/(discount*forward)"))\n    print("Profile:",profile,"| device:",jax.devices(),"| output:",out)\n    return cfg\n\ndef basis(x, M=4):\n    centres=np.array([-.6,-.2,.2,.6]) if M==4 else np.linspace(-.6,.6,M)\n    width=.45 if M==4 else 1.8/max(M-1,1)\n    z=(np.asarray(x)[None,:]-centres[:,None])/width\n    phi=.04*(1+np.cos(np.pi*np.clip(z,-1,1)))*(np.abs(z)<=1)\n    return phi\n\ndef curvature_matrix(K):\n    K=np.asarray(K); dm=np.diff(K)[:-1]; dp=np.diff(K)[1:]; N=len(K)-2\n    B=np.zeros((N,N+2))\n    ii=np.arange(N)\n    B[ii,ii]=2/(dm*(dm+dp)); B[ii,ii+1]=-2/(dm*dp)\n    B[ii,ii+2]=2/(dp*(dm+dp))\n    return B\n\n@lru_cache(maxsize=32)\ndef make_grid(n, M=4, xmin=-1.2, xmax=1.2):\n    N=2**n; h=(xmax-xmin)/(N+1)\n    xe=np.linspace(xmin,xmax,N+2); K=np.exp(xe); x=xe[1:-1]\n    kap=2*np.tanh(h/2)/h\n    Lt=np.diag(np.full(N,2/h**2))\n    Lt+=np.diag(np.full(N-1,-1/h**2+kap/(2*h)),1)\n    Lt+=np.diag(np.full(N-1,-1/h**2-kap/(2*h)),-1)\n    beta=(1-K[0])/(K[0]-K[-1]); G=-beta*K[-1]+beta*K\n    phi=basis(x,M)\n    A0=.005*Lt; Am=.5*phi[:,:,None]*Lt[None,:,:]\n    cbar=sum(max(0.,-sla.eigvalsh((a+a.T)/2,subset_by_index=[0,0])[0])\n             for a in [A0,*Am])\n    return dict(n=n,N=N,M=M,h=h,x=x,xe=xe,K=K,G=G,phi=phi,\n                Lt=Lt,A0=A0,Am=Am,cbar=float(cbar),\n                v0=np.maximum(1-K[1:-1],0)-G[1:-1],B=curvature_matrix(K),\n                xmin=xmin,xmax=xmax)\n\ndef assemble(g, xi): return g["A0"]+np.einsum("m,mij->ij",np.asarray(xi),g["Am"])\ndef sigma2(g, xi): return .01+np.asarray(xi)@g["phi"]\n\ndef interpolation(K, query):\n    """Piecewise linear in strike, never log-strike; no snapping or clipping."""\n    query=np.asarray(query)\n    if np.any(query<K[0]) or np.any(query>K[-1]):\n        raise ValueError("Query outside the declared strike domain")\n    left=np.clip(np.searchsorted(K,query,side="right")-1,0,len(K)-2)\n    theta=(query-K[left])/(K[left+1]-K[left])\n    W=np.zeros((len(query),len(K))); rr=np.arange(len(query))\n    W[rr,left]=1-theta; W[rr,left+1]=theta\n    return W\n\ndef price_vector(g,v): return np.r_[g["G"][0],np.asarray(v)+g["G"][1:-1],g["G"][-1]]\n\ndef audit_prices(K,Cs,tol=1e-8):\n    """All metrics retain units; density tolerance is derived, not reused blindly."""\n    Cs=np.asarray(Cs); dm=np.diff(K)[:-1]; dp=np.diff(K)[1:]\n    butterfly=(dm*Cs[:,2:]+dp*Cs[:,:-2])/(dm+dp)-Cs[:,1:-1]\n    density=2*butterfly/(dm*dp)\n    calendar=np.diff(np.vstack([np.maximum(1-K,0),Cs]),axis=0)\n    slope=np.diff(Cs,axis=1)/np.diff(K)\n    bounds=np.minimum(Cs-np.maximum(1-K,0),1-Cs)\n    return dict(butterfly_min=float(butterfly.min()),density_min=float(density.min()),\n                calendar_min=float(calendar.min()),vertical_min=float(np.minimum(-slope,1+slope).min()),\n                price_bound_min=float(bounds.min()),\n                butterfly_violation=float(max(0,-butterfly.min())),\n                density_violation=float(max(0,-density.min())),\n                calendar_violation=float(max(0,-calendar.min())),\n                bfly_count=int((butterfly < -tol).sum()),\n                cal_count=int((calendar < -tol).sum()),\n                audit_price_tolerance=tol)\n\ndef certificate(g, approx, exact, roundoff=2e-10):\n    """Conditional certificate given a measured absolute price error, not a priori LCHS proof."""\n    ap=np.asarray(approx); ex=np.asarray(exact)\n    delta=np.max(np.abs(ap-ex),axis=1)\n    dm=np.diff(g["K"])[:-1]; dp=np.diff(g["K"])[1:]\n    b=(dm*ap[:,2:]+dp*ap[:,:-2])/(dm+dp)-ap[:,1:-1]\n    den=2*b/(dm*dp)\n    cal=np.diff(np.vstack([np.maximum(1-g["K"],0),ap]),axis=0)\n    cb=delta+np.r_[0,delta[:-1]]\n    ok=(np.all(b>=-2*delta[:,None]-roundoff) and\n        np.all(den>=-4*delta[:,None]/(dm*dp)-roundoff*2/(dm*dp)) and\n        np.all(cal>=-cb[:,None]-roundoff))\n    return dict(price_error_inf=float(delta.max()),butterfly_bound=float(2*delta.max()),\n                calendar_bound=float(cb.max()),density_bound=float((4*delta[:,None]/(dm*dp)).max()),\n                certificate_pass=bool(ok))\n\ndef lchs_weights(kernel="beta", terms=512, cutoff=48., beta=.8):\n    """Symmetric midpoint rule makes conjugate pairing exact up to rounding.\n    Normalisation enforces the identity at t=0; it does not recover omitted tails.\n    """\n    k=-cutoff+(np.arange(terms)+.5)*2*cutoff/terms\n    f=1/(np.pi*(1+1j*k)) if kernel=="simple" else np.exp(2**beta-(1+1j*k)**beta)/(2*np.pi)\n    w=(2*cutoff/terms)*f/(1-1j*k); total=w.sum()\n    if abs(total)<1e-8: raise ValueError("Unstable quadrature normalisation")\n    return k,w/total,dict(terms=terms,cutoff=cutoff,dk=2*cutoff/terms,\n                          normalisation_defect=float(abs(total-1)),alpha_lcu=float(np.abs(w/total).sum()))\n\n@jax.custom_jvp\ndef hermitian_unitary(X):\n    lam,V=jnp.linalg.eigh(X)\n    return (V*jnp.exp(-1j*lam)[None,:])@V.conj().T\n\n@hermitian_unitary.defjvp\ndef _hermitian_unitary_jvp(primals,tangents):\n    (X,),(dX,)=primals,tangents\n    lam,V=jnp.linalg.eigh(X)\n    U=(V*jnp.exp(-1j*lam)[None,:])@V.conj().T\n    diff=lam[:,None]-lam[None,:]; avg=(lam[:,None]+lam[None,:])/2\n    divided=-1j*jnp.exp(-1j*avg)*jnp.sinc(diff/(2*jnp.pi))\n    dU=V@(divided*(V.conj().T@dX@V))@V.conj().T\n    return U,dU\n\ndef lchs_apply_jax(A,v,dt,k,w,shift,chunk=16):\n    As=A+shift*jnp.eye(A.shape[0]); L=(As+As.T)/2; H=-.5j*(As-As.T)\n    pad=(-len(k))%chunk\n    kk=jnp.pad(jnp.asarray(k),(0,pad)).reshape(-1,chunk)\n    ww=jnp.pad(jnp.asarray(w),(0,pad)).reshape(-1,chunk)\n    def part(carry,kw):\n        kp,wp=kw\n        U=jax.vmap(hermitian_unitary)(dt*(H[None,:,:]+kp[:,None,None]*L))\n        return carry+jnp.einsum("k,kij,j->i",wp,U,v),None\n    val,_=jax.lax.scan(jax.checkpoint(part),jnp.zeros(v.shape,dtype=jnp.complex128),(kk,ww))\n    return jnp.exp(shift*dt)*jnp.real(val)\n\n@lru_cache(maxsize=48)\ndef solver_functions(n,M,method,terms,cutoff,dt,fd_steps=256,xmin=-1.2,xmax=1.2):\n    g=make_grid(n,M,xmin,xmax)\n    if method=="fd": return None\n    A0=jnp.asarray(g["A0"]); Am=jnp.asarray(g["Am"]); shift=g["cbar"]\n    if method.startswith("lchs"):\n        k,w,_=lchs_weights(method.split("-")[1],terms,cutoff)\n    def forward(xi,v):\n        A=A0+jnp.einsum("m,mij->ij",xi,Am)\n        if method=="expm": return jax.scipy.linalg.expm(-dt*A,max_squarings=24)@v\n        return lchs_apply_jax(A,v,dt,k,w,shift)\n    def objective(xi,v,W,y,weight,lam):\n        residual=(W@forward(xi,v)-y)*weight\n        smooth=jnp.diff(xi,n=2)\n        return jnp.mean(residual**2)+lam*jnp.mean(smooth**2) if M>2 else jnp.mean(residual**2)\n    return jax.jit(forward),jax.jit(jax.value_and_grad(objective))\n\ndef fd_value_grad(g,xi,v0,dt,W,y,weight,lam,steps):\n    """Backward Euler, one sparse LU, exact discrete adjoint of the implemented loss."""\n    A=assemble(g,xi); dh=dt/steps\n    fac=spla.splu(sp.eye(g["N"],format="csc")+dh*sp.csc_matrix(A))\n    trajectory=[np.asarray(v0)]\n    for _ in range(steps): trajectory.append(fac.solve(trajectory[-1]))\n    v=trajectory[-1]; r=W@v-y\n    value=float(np.mean((weight*r)**2)); adj=2*W.T@(weight**2*r)/len(y)\n    grad=np.zeros(g["M"])\n    for state in reversed(trajectory[1:]):\n        adj=fac.solve(adj,trans="T")\n        grad-=dh*.5*(g["phi"]@(adj*(g["Lt"]@state)))\n    if g["M"]>2:\n        D=np.diff(np.eye(g["M"]),n=2,axis=0); d=D@xi\n        value+=lam*np.mean(d*d); grad+=2*lam*D.T@d/len(d)\n    return value,grad,v\n\ndef normalised_call(k,T,vol):\n    if T<=0: return np.maximum(1-np.asarray(k),0)\n    s=np.asarray(vol)*np.sqrt(T)\n    d1=-np.log(k)/s+s/2\n    return ndtr(d1)-np.asarray(k)*ndtr(d1-s)\n\ndef implied_vol(price,k,T):\n    lo=max(1-k,0)\n    if not np.isfinite(price) or price<=lo+1e-12 or price>=1-1e-12: return np.nan\n    try: return brentq(lambda v:normalised_call(k,T,v)-price,1e-6,8.,xtol=1e-12)\n    except ValueError: return np.nan\n\ndef true_sigma2(x,s,truth):\n    if truth=="in": return .01+XI_TRUTH[min(s,1)]@basis(np.asarray(x),4)\n    return .01+.055*np.exp(-((np.asarray(x)+.1)/.3)**2)+.035/(1+np.exp(4*np.asarray(x)))\n\n@lru_cache(maxsize=12)\ndef reference_surface(truth,ref_n,times_tuple=tuple(TIMES),xmin=-1.2,xmax=1.2):\n    """Fixed fine strike-space Dupire discretisation, rho=1, sparse exponential action."""\n    N=2**ref_n; K=np.exp(np.linspace(xmin,xmax,N+2)); B=curvature_matrix(K)\n    beta=(1-K[0])/(K[0]-K[-1]); G=beta*(K-K[-1]); v=np.maximum(1-K[1:-1],0)-G[1:-1]\n    Cs=[]; previous=0.\n    for T in times_tuple:\n        stops=([float(TIMES[0])] if previous<TIMES[0]<T else [])+[T]\n        for stop in stops:\n            s=int(previous>=TIMES[0]-1e-12)\n            a=.5*true_sigma2(np.log(K[1:-1]),s,truth)*K[1:-1]**2\n            Q=sp.diags(a)@sp.csc_matrix(B[:,1:-1])\n            v=spla.expm_multiply((stop-previous)*Q,v,traceA=float((stop-previous)*Q.diagonal().sum()))\n            previous=stop\n        Cs.append(np.r_[G[0],v+G[1:-1],G[-1]])\n    return K,np.asarray(Cs)\n\ndef make_dataset(truth,noise_bp,noise_seed,ref_n):\n    K,Cs=reference_surface(truth,ref_n); rng=np.random.default_rng(noise_seed)\n    quotes=[]\n    for s,T in enumerate(TIMES):\n        ks=TRAIN_K[s]; exact=interpolation(K,ks)@Cs[s]\n        perturb=rng.uniform(-noise_bp*1e-4,noise_bp*1e-4,len(ks))\n        mid=exact+perturb; half=np.full(len(ks),noise_bp*1e-4)\n        # Do not clip inconsistent noisy observations into the feasible set.\n        quotes.append(dict(T=float(T),k=ks,mid=mid,bid=mid-half,ask=mid+half,clean=exact))\n    kid=np.linspace(.7,1.4,181)\n    tests=[]\n    for s,q in enumerate(quotes):\n        mask=~np.any(np.isclose(kid[:,None],q["k"][None,:],atol=1e-10,rtol=0),axis=1)\n        kh=kid[mask]; tests.append(dict(T=q["T"],k=kh,price=interpolation(K,kh)@Cs[s]))\n    return dict(truth=truth,noise_bp=noise_bp,noise_seed=noise_seed,quotes=quotes,tests=tests,\n                dataset_id=f"{truth}_bp{noise_bp:g}_d{noise_seed}",kind="synthetic")\n\ndef calibrate_model(g,data,method,cfg,seed=0,lam=0.,start=None):\n    rng=np.random.default_rng(seed); v=g["v0"].copy(); hats=[]; outputs=[]; traces=[]\n    compilation=0.; opt_time=0.; forward_calls=0; previous=0.; status=[]\n    for s,q in enumerate(data["quotes"]):\n        dt=q["T"]-previous; previous=q["T"]\n        Wfull=interpolation(g["K"],q["k"]); W=Wfull[:,1:-1]\n        y=q["mid"]-Wfull@g["G"]\n        weight=np.ones(len(y))\n        if data.get("kind")=="market":\n            weight=1/np.maximum((q["ask"]-q["bid"])/2,1e-5)\n        z0=np.clip(.5+.15*rng.standard_normal(g["M"]),0,1) if start is None else np.array(start[s])\n        args=(jnp.asarray(v),jnp.asarray(W),jnp.asarray(y),jnp.asarray(weight),float(lam))\n        if method=="fd":\n            valuegrad=lambda z:fd_value_grad(g,z,v,dt,W,y,weight,lam,cfg["fd_steps"])[:2]\n            t=time.perf_counter(); valuegrad(z0); warm=time.perf_counter()-t\n        else:\n            cutoff=64. if method=="lchs-simple" else 48.\n            fwd,vg=solver_functions(g["n"],g["M"],method,cfg["lchs_terms"],cutoff,float(dt),\n                                     cfg["fd_steps"],g["xmin"],g["xmax"])\n            t=time.perf_counter(); jax.block_until_ready(vg(jnp.asarray(z0),*args)); warm=time.perf_counter()-t\n            valuegrad=lambda z:tuple(np.asarray(x) for x in vg(jnp.asarray(z),*args))\n        compilation+=warm; records=[]\n        def objective(z):\n            value,gradient=valuegrad(z); records.append(float(value))\n            if not np.isfinite(value) or not np.all(np.isfinite(gradient)):\n                raise FloatingPointError("Non-finite calibration loss or gradient")\n            return float(value),np.asarray(gradient,dtype=float)\n        t=time.perf_counter()\n        result=minimize(objective,z0,jac=True,bounds=[(0,1)]*g["M"],method="L-BFGS-B",\n                        options=dict(maxiter=cfg["maxiter"],ftol=1e-13,gtol=1e-9,maxls=30))\n        opt_time+=time.perf_counter()-t; forward_calls+=len(records)\n        hats.append(result.x); traces.append(records); status.append(bool(result.success))\n        if method=="fd": v=fd_value_grad(g,result.x,v,dt,W,y,weight,lam,cfg["fd_steps"])[2]\n        else: v=np.asarray(fwd(jnp.asarray(result.x),jnp.asarray(v)))\n        outputs.append(price_vector(g,v))\n    exact=[]; v=g["v0"].copy(); previous=0.\n    for xi,q in zip(hats,data["quotes"]):\n        v=sla.expm(-(q["T"]-previous)*assemble(g,xi))@v; previous=q["T"]\n        exact.append(price_vector(g,v))\n    return dict(hats=np.array(hats),C=np.array(outputs),C_exact=np.array(exact),\n                trace=traces,converged=all(status),slice_converged=status,\n                compile_warm_s=compilation,opt_s=opt_time,nfev=forward_calls)\n\ndef evaluate_model(g,data,result,method,seed,lam):\n    rs=[]; held=[]; ivs=[]; total=0; validiv=0; inside=[]; invalid_obs=0\n    Cs=result["C"]\n    for s,q in enumerate(data["quotes"]):\n        p=(np.asarray(result["quote_predictions"][s]) if "quote_predictions" in result\n           else interpolation(g["K"],q["k"])@Cs[s]); rs.extend(p-q["mid"])\n        inside.extend((p>=q["bid"]-1e-10)&(p<=q["ask"]+1e-10))\n        invalid_obs+=int(((q["mid"]<np.maximum(1-q["k"],0))|(q["mid"]>1)).sum())\n        test=data["tests"][s]\n        ph=(np.asarray(result["test_predictions"][s]) if "test_predictions" in result\n            else interpolation(g["K"],test["k"])@Cs[s])\n        held.extend(ph-test["price"])\n        for k,mod,ref in zip(test["k"],ph,test["price"]):\n            a=implied_vol(mod,k,q["T"]); b=implied_vol(ref,k,q["T"]); total+=1\n            if np.isfinite(a) and np.isfinite(b): ivs.append((a-b)**2); validiv+=1\n    aud=audit_prices(g["K"],Cs)\n    row=dict(dataset=data["dataset_id"],truth=data["truth"],noise_bp=data["noise_bp"],\n             noise_seed=data["noise_seed"],method=method,n=g["n"],N=g["N"],M=g["M"],\n             seed=seed,lam=lam,rmse_quotes=float(np.sqrt(np.mean(np.square(rs)))),\n             rmse_heldout=float(np.sqrt(np.mean(np.square(held)))),\n             iv_rmse=float(np.sqrt(np.mean(ivs))) if ivs else np.nan,\n             iv_valid=validiv,iv_total=total,bidask_fraction=float(np.mean(inside)),\n             invalid_observations=invalid_obs,nfev=result.get("nfev",0),\n             compile_warm_s=result.get("compile_warm_s",0),opt_s=result.get("opt_s",0),\n             converged=result.get("converged",False),**aud)\n    if "C_exact" in result:\n        row.update(certificate(g,Cs,result["C_exact"]))\n        row.update({"exact_"+k:v for k,v in audit_prices(g["K"],result["C_exact"]).items()})\n    if "hats" in result and data["kind"]=="synthetic":\n        covered=(g["K"][1:-1]>=.7)&(g["K"][1:-1]<=1.4)\n        row["sigma2_linf"]=float(max(np.max(np.abs(sigma2(g,h)-true_sigma2(g["x"],s,data["truth"]))[covered])\n                                    for s,h in enumerate(result["hats"])))\n    return row\n', 'benchmarks.py': '"""Benchmark and diagnostic cells. Requires core.py definitions in this namespace."""\nimport QuantLib as ql\n\ndef ah_calibrate(g,data,cfg):\n    today=ql.Date(1,1,2025); ql.Settings.instance().evaluationDate=today\n    dc=ql.Actual365Fixed(); zero=ql.YieldTermStructureHandle(ql.FlatForward(today,0.,dc))\n    quotes=ql.CalibrationSet()\n    for q in data["quotes"]:\n        days=int(round(q["T"]*365))\n        if abs(days/365-q["T"])>1e-9:\n            raise ValueError("AH requires ACT/365F maturities, no silent date rounding")\n        for k,price in zip(q["k"],q["mid"]):\n            vol=implied_vol(price,k,q["T"])\n            if not np.isfinite(vol):\n                raise ValueError("AH input has no finite Black IV; quotes retained, baseline marked failed")\n            option=ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call,float(k)),ql.EuropeanExercise(today+days))\n            quotes.push_back((option,ql.SimpleQuote(float(vol))))\n    t=time.perf_counter()\n    ah=ql.AndreasenHugeVolatilityInterpl(quotes,ql.QuoteHandle(ql.SimpleQuote(1.)),zero,zero,\n                ql.AndreasenHugeVolatilityInterpl.Linear,ql.AndreasenHugeVolatilityInterpl.Call,\n                max(200,2*g["N"]),float(g["K"][0]*.8),float(g["K"][-1]*1.2))\n    error=ah.calibrationError()  # force the native calibration inside the timer\n    elapsed=time.perf_counter()-t\n    Cs=np.array([[ah.optionPrice(q["T"],float(k),ql.Option.Call) for k in g["K"]]\n                 for q in data["quotes"]])\n    qp=[[ah.optionPrice(q["T"],float(k),ql.Option.Call) for k in q["k"]] for q in data["quotes"]]\n    hp=[[ah.optionPrice(q["T"],float(k),ql.Option.Call) for k in q["k"]] for q in data["tests"]]\n    return dict(C=Cs,quote_predictions=qp,test_predictions=hp,\n                converged=True,opt_s=elapsed,compile_warm_s=0.,nfev=0,\n                native_error=str(error),method_note="QuantLib Andreasen-Huge linear interpolation")\n\ndef init_mlp(key,width):\n    sizes=[2,width,width,1]; pars=[]\n    for a,b in zip(sizes[:-1],sizes[1:]):\n        key,sub=jax.random.split(key)\n        pars.append((jax.random.normal(sub,(a,b))*np.sqrt(2/(a+b)),jnp.zeros(b)))\n    return pars\n\ndef mlp(pars,x):\n    for w,b in pars[:-1]: x=jnp.tanh(x@w+b)\n    return (x@pars[-1][0]+pars[-1][1]).squeeze()\n\ndef pinn_calibrate(g,data,cfg,seed=0,adaptive=True):\n    """Wam-style adaptation, NOT a numerical reproduction of CIFEr 2024.\n    The original alternating SA / auto-balancing schedule is retained. Two\n    independent networks make the PDE residual non-tautological. See README.\n    """\n    key=jax.random.PRNGKey(seed); k1,k2=jax.random.split(key)\n    tree=(init_mlp(k1,cfg["pinn_width"]),init_mlp(k2,cfg["pinn_width"]))\n    theta,unravel=ravel_pytree(tree)\n    Tmax=data["quotes"][-1]["T"]\n    kt=np.concatenate([q["k"] for q in data["quotes"]]); tt=np.concatenate([np.full(len(q["k"]),q["T"]) for q in data["quotes"]])\n    yt=jnp.array(np.concatenate([q["mid"] for q in data["quotes"]])); train=jnp.array(np.c_[kt,tt])\n    quote_weight=jnp.array(np.concatenate([1/np.maximum((q["ask"]-q["bid"])/2,1e-5)\n                 if data["kind"]=="market" else np.ones(len(q["k"])) for q in data["quotes"]]))\n    rng=np.random.default_rng(919+seed)\n    xc=np.exp(rng.uniform(g["xmin"],g["xmax"],cfg["collocation"]))\n    tc=rng.uniform(.02*Tmax,Tmax,cfg["collocation"])\n    coll=jnp.array(np.c_[xc,tc]); boundary=jnp.array(np.c_[np.tile(g["K"][[0,-1]],24),np.repeat(np.linspace(.02*Tmax,Tmax,24),2)])\n    def functions(th,z):\n        p,lv=unravel(th); k,t=z\n        inp=jnp.array([jnp.log(k)/1.2,2*t/Tmax-1])\n        iv=.03+.35*jax.nn.softplus(mlp(p,inp))\n        s=iv*jnp.sqrt(jnp.maximum(t,1e-10)); d=-jnp.log(k)/s+s/2\n        c=jax.scipy.special.ndtr(d)-k*jax.scipy.special.ndtr(d-s)\n        variance=.01+.04*jax.nn.softplus(mlp(lv,inp))\n        return c,variance\n    def cfun(th,z): return functions(th,z)[0]\n    def errors(th):\n        pred=jax.vmap(cfun,(None,0))(th,train)\n        vals=jax.vmap(functions,(None,0))(th,coll)\n        deriv=jax.vmap(jax.grad(cfun,argnums=1),(None,0))(th,coll)\n        hess=jax.vmap(jax.hessian(cfun,argnums=1),(None,0))(th,coll)\n        ck,ct=deriv[:,0],deriv[:,1]; ckk=hess[:,0,0]\n        pde=ct-.5*vals[1]*coll[:,0]**2*ckk\n        cb=jax.vmap(cfun,(None,0))(th,boundary)\n        btarget=jnp.maximum(1-boundary[:,0],0)\n        return (((pred-yt)*quote_weight)**2,pde**2,jax.nn.relu(-ckk)**2,\n                jax.nn.relu(-ct)**2,(jax.nn.relu(ck)**2+jax.nn.relu(-1-ck)**2),\n                (cb-btarget)**2)\n    sa=tuple(jnp.ones(len(x)) for x in errors(theta)); weights=jnp.ones(6)\n    def components(th,sa,weights): return jnp.stack([w*jnp.mean(a*e) for w,a,e in zip(weights,sa,errors(th))])\n    @jax.jit\n    def update(th,m,v,step,sa,weights):\n        loss,gr=jax.value_and_grad(lambda p:jnp.sum(components(p,sa,weights)))(th)\n        m=.9*m+.1*gr; v=.999*v+.001*gr**2\n        lr=1e-3*.9**(step/2000)\n        th=th-lr*(m/(1-.9**(step+1)))/(jnp.sqrt(v/(1-.999**(step+1)))+1e-8)\n        return th,m,v,loss\n    @jax.jit\n    def rebalance(th,sa,w):\n        jac=jax.jacrev(components)(th,sa,w)\n        norms=jnp.mean(jnp.abs(jac),axis=1)\n        target=jnp.where(norms>1e-15,jnp.sum(norms)/jnp.maximum(norms,1e-15),1.)\n        return jax.lax.stop_gradient(.9*w+.1*jnp.clip(target,1e-3,1e3))\n    @jax.jit\n    def ascend(th,sa,w):\n        # Gradient ascent of weighted mean loss w.r.t. per-point multipliers.\n        return tuple(jnp.minimum(a+weight*err/len(err),1e3) for a,weight,err in zip(sa,w,errors(th)))\n    m=jnp.zeros_like(theta); v=m; hist=[]\n    t=time.perf_counter(); jax.block_until_ready(update(theta,m,v,0,sa,weights))\n    if adaptive:\n        jax.block_until_ready(rebalance(theta,sa,weights)); jax.block_until_ready(ascend(theta,sa,weights))\n    warm=time.perf_counter()-t; t=time.perf_counter()\n    for step in range(cfg["pinn_steps"]):\n        if adaptive and step%100==0: weights=rebalance(theta,sa,weights)\n        if adaptive and (step+50)%100==0: sa=ascend(theta,sa,weights)\n        theta,m,v,loss=update(theta,m,v,step,sa,weights)\n        if step%100==0 or step==cfg["pinn_steps"]-1:\n            comp=np.asarray(components(theta,tuple(jnp.ones_like(a) for a in sa),jnp.ones(6)))\n            hist.append(dict(step=step,loss=float(loss),fit=comp[0],pde=comp[1],butterfly=comp[2],calendar=comp[3]))\n    jax.block_until_ready(theta); elapsed=time.perf_counter()-t\n    if not np.all(np.isfinite(theta)): raise FloatingPointError("PINN non-finite parameters")\n    predict=jax.jit(jax.vmap(cfun,(None,0)))\n    Cs=np.array([predict(theta,jnp.array(np.c_[g["K"],np.full(len(g["K"]),q["T"])])) for q in data["quotes"]])\n    qp=[np.asarray(predict(theta,jnp.array(np.c_[q["k"],np.full(len(q["k"]),q["T"])]))) for q in data["quotes"]]\n    hp=[np.asarray(predict(theta,jnp.array(np.c_[q["k"],np.full(len(q["k"]),q["T"])]))) for q in data["tests"]]\n    return dict(C=Cs,quote_predictions=qp,test_predictions=hp,\n                converged=False,training_completed=True,opt_s=elapsed,compile_warm_s=warm,\n                nfev=cfg["pinn_steps"],pinn_history=hist,weights=np.asarray(weights),\n                parameter_count=len(theta),method_note="Fixed epoch budget; convergence not asserted")\n\ndef run_calibration_suite(cfg):\n    allrows=[]; failures=[]; selected={}; out=cfg["out"]\n    cases=[(truth,bp,ds) for truth in ["in","out"] for bp in [0.,5.]\n           for ds in ([101] if bp==0 else cfg["noise_seeds"])]\n    comparison_n=cfg["ns"][min(1,len(cfg["ns"])-1)]\n    for truth,bp,ds in cases:\n        data=make_dataset(truth,bp,ds,cfg["ref_n"])\n        save_json(out/"raw"/(data["dataset_id"]+".json"),data)\n        for n in cfg["ns"]:\n            g=make_grid(n)\n            methods=["expm","fd","lchs-beta","lchs-simple"]\n            if n==comparison_n: methods += ["AH","PINN-fixed","PINN-Wam"]\n            for method in methods:\n                seeds=[cfg["seeds"][0]] if method=="AH" else cfg["seeds"]\n                for seed in seeds:\n                    lam=0.  # core solver comparison uses exactly the same objective\n                    key=f\'{data["dataset_id"]}_n{n}_{method}_s{seed}\'\n                    cp=out/"checkpoints"/(key+".json")\n                    if cp.exists():\n                        payload=json.loads(cp.read_text()); allrows.append(payload["row"])\n                        if seed==cfg["seeds"][0] and n==comparison_n:\n                            selected[key]=payload\n                        continue\n                    fp=out/"checkpoints"/(key+"_failure.json")\n                    if fp.exists():\n                        failures.append(json.loads(fp.read_text())); continue\n                    try:\n                        if method=="AH": result=ah_calibrate(g,data,cfg)\n                        elif method.startswith("PINN"):\n                            result=pinn_calibrate(g,data,cfg,seed,adaptive=(method=="PINN-Wam"))\n                        else: result=calibrate_model(g,data,method,cfg,seed,lam)\n                        row=evaluate_model(g,data,result,method,seed,lam)\n                        row["profile"]=cfg["profile"]; row["kind"]="synthetic"\n                        payload=dict(row=row,result=result)\n                        save_json(cp,payload); allrows.append(row)\n                        if seed==cfg["seeds"][0] and n==comparison_n: selected[key]=jsonable(payload)\n                        print(f\'{key}: quote {row["rmse_quotes"]:.3e}, test {row["rmse_heldout"]:.3e}, opt {row["opt_s"]:.1f}s\')\n                    except (ValueError,FloatingPointError,RuntimeError) as exc:\n                        failure=dict(key=key,method=method,dataset=data["dataset_id"],reason=str(exc))\n                        save_json(fp,failure)\n                        failures.append(failure); print("FAILED:",failure)\n                    pd.DataFrame(allrows).to_csv(out/"results.csv",index=False)\n                    save_json(out/"failures.json",failures)\n    pd.DataFrame(allrows).to_csv(out/"results.csv",index=False)\n    save_json(out/"failures.json",failures)\n    save_json(out/"selected_examples.json",selected)\n    return pd.DataFrame(allrows),selected\n\ndef finite_difference_grad(fun,x,eps=1e-5):\n    eye=np.eye(len(x))*eps\n    return np.array([(fun(x+d)-fun(x-d))/(2*eps) for d in eye])\n\ndef run_validation(cfg):\n    rows=[]\n    for n in cfg["ns"]:\n        g=make_grid(n); N=g["N"]; h=g["h"]\n        rho=4*np.sinh(h/2)**2/h**2\n        operator=rho*g["K"][1:-1,None]**2*g["B"]\n        assert np.allclose(operator[:,1:-1],-g["Lt"],atol=1e-8,rtol=1e-11)\n        lift=float(np.max(np.abs(operator@g["G"])))\n        rng=np.random.default_rng(901); worst=0.; positive_shift=None\n        for xi in np.r_[rng.uniform(0,1,(16,4)),np.eye(4),1-np.eye(4)]:\n            A=assemble(g,xi); c=max(0.,-sla.eigvalsh((A+A.T)/2)[0]); worst=max(worst,c)\n            if c>1e-5: positive_shift=xi\n            off=-A+np.diag(np.diag(A)); assert np.min(off)>=-1e-12\n            density_generator=g["B"][:,1:-1]@np.diag(rho*.5*sigma2(g,xi)*g["K"][1:-1]**2)\n            assert np.min(density_generator-np.diag(np.diag(density_generator)))>=-1e-10\n            assert c<=g["cbar"]+1e-10\n        xi=np.array([.4,.6,.5,.3]); data=make_dataset("in",0.,101,cfg["ref_n"])\n        q=data["quotes"][0]; Wf=interpolation(g["K"],q["k"]); W=Wf[:,1:-1]; y=q["mid"]-Wf@g["G"]\n        val,grad,_=fd_value_grad(g,xi,g["v0"],q["T"],W,y,np.ones(len(y)),0.,cfg["fd_steps"])\n        fd=finite_difference_grad(lambda z:fd_value_grad(g,z,g["v0"],q["T"],W,y,np.ones(len(y)),0.,cfg["fd_steps"])[0],xi)\n        fd_error=float(np.linalg.norm(grad-fd)/max(1e-12,np.linalg.norm(fd)))\n        assert fd_error<1e-4\n        fw,vg=solver_functions(n,4,"lchs-beta",cfg["lchs_terms"],48.,float(q["T"]))\n        args=tuple(jnp.array(x) for x in [g["v0"],W,y,np.ones(len(y))])+(0.,)\n        _,grad=vg(jnp.array(xi),*args)\n        fd=finite_difference_grad(lambda z:float(vg(jnp.array(z),*args)[0]),xi)\n        lerr=float(np.linalg.norm(np.asarray(grad)-fd)/max(1e-12,np.linalg.norm(fd)))\n        assert lerr<3e-4\n        # Constant-vector and lift consistency in strike-space observation.\n        assert np.allclose(Wf.sum(1),1) and np.allclose(Wf@g["K"],q["k"])\n        rows.append(dict(n=n,lift_residual=lift,fitted_identity_residual=float(np.max(np.abs(operator[:,1:-1]+g["Lt"]))),\n                         fd_gradient_relative_error=fd_error,lchs_gradient_relative_error=lerr,\n                         maximum_sampled_shift=worst,uniform_shift=g["cbar"],\n                         positive_shift_sample=positive_shift is not None))\n    # Reference refinement at unchanged quotes/tests, not changing the data per N.\n    for truth in ["in","out"]:\n        K,C=reference_surface(truth,cfg["ref_n"]); K2,C2=reference_surface(truth,cfg["ref_n"]+1)\n        pts=np.linspace(.7,1.4,101)\n        err=float(np.max(np.abs(C@interpolation(K,pts).T-C2@interpolation(K2,pts).T)))\n        rows.append(dict(test="reference_refinement",truth=truth,reference_price_inf=err))\n    frame=pd.DataFrame(rows); frame.to_csv(cfg["out"]/"validation.csv",index=False)\n    print("Structural identities, fixed quote operators, and implemented gradients: PASS")\n    return frame\n\ndef run_error_budget(cfg):\n    rows=[]; rng=np.random.default_rng(707)\n    for n in cfg["ns"]:\n        g=make_grid(n); xi=rng.uniform(.15,.85,(2,4))\n        exact=[]; v=g["v0"].copy(); prior=0.\n        for s,T in enumerate(TIMES):\n            v=sla.expm(-(T-prior)*assemble(g,xi[s]))@v; exact.append(price_vector(g,v)); prior=T\n        for kernel in ["simple","beta"]:\n            for terms in cfg["kernels"]:\n                for cutoff in ([16.,32.,64.] if cfg["profile"]!="smoke" else [8.,24.]):\n                    k,w,meta=lchs_weights(kernel,terms,cutoff)\n                    v=jnp.array(g["v0"]); approx=[]; previous=0.\n                    call=jax.jit(lambda A,v,dt:lchs_apply_jax(A,v,dt,k,w,g["cbar"]))\n                    t=time.perf_counter(); jax.block_until_ready(call(jnp.array(assemble(g,xi[0])),v,TIMES[0])); warm=time.perf_counter()-t\n                    t=time.perf_counter()\n                    for s,T in enumerate(TIMES):\n                        v=call(jnp.array(assemble(g,xi[s])),v,T-previous); previous=T\n                        approx.append(price_vector(g,v))\n                    jax.block_until_ready(v); elapsed=time.perf_counter()-t\n                    cert=certificate(g,approx,exact)\n                    assert cert["certificate_pass"],"Error-bound check failed"\n                    row=dict(n=n,kernel=kernel,**meta,**cert,**audit_prices(g["K"],approx),\n                             solver_s=elapsed,compile_warm_s=warm,shift=g["cbar"],\n                             simulation_time_proxy=cutoff*sum(TIMES-np.r_[0,TIMES[:-1]])*np.linalg.norm(assemble(g,xi[0]),2))\n                    rows.append(row)\n        pd.DataFrame(rows).to_csv(cfg["out"]/"error_budget.csv",index=False)\n    return pd.DataFrame(rows)\n\ndef run_diagnostics(cfg,selected):\n    rows=[]; n=cfg["ns"][min(1,len(cfg["ns"])-1)]; g=make_grid(n)\n    data=make_dataset("in",0.,101,cfg["ref_n"])\n    key=f\'{data["dataset_id"]}_n{n}_expm_s{cfg["seeds"][0]}\'\n    hats=np.asarray(selected[key]["result"]["hats"])\n    for bits in [2,4,6,8,10,12]:\n        quant=np.round(hats*(2**bits-1))/(2**bits-1)\n        exact=[]; v=g["v0"].copy(); previous=0.\n        for h,q in zip(quant,data["quotes"]):\n            v=sla.expm(-(q["T"]-previous)*assemble(g,h))@v; previous=q["T"]\n            exact.append(price_vector(g,v))\n        result=dict(hats=quant,C=np.array(exact),C_exact=np.array(exact),converged=True)\n        row=evaluate_model(g,data,result,"quantised-expm",0,0.)\n        row.update(bits=bits,design_qubits=bits*g["M"],hxi=1/(2**bits-1),\n                   inverse_hxi_squared=(2**bits-1)**2,continuous_reference=selected[key]["row"]["rmse_quotes"])\n        rows.append(row)\n    pd.DataFrame(rows).to_csv(cfg["out"]/"quantisation.csv",index=False)\n    # Smoothness: global Fourier approximation of the compactly supported bump.\n    x=np.linspace(-1.2,1.2,4096,endpoint=False); phi=basis(x)[1]; ft=np.fft.fft(phi)/len(x)\n    freqs=np.fft.fftfreq(len(x))*len(x); fourier=[]\n    for degree in [2,4,8,16,32,64,128]:\n        approx=np.fft.ifft(ft*(np.abs(freqs)<=degree)*len(x)).real\n        fourier.append(dict(degree=degree,error_inf=float(np.max(np.abs(approx-phi))),\n                            minimum_basis=float(approx.min()),variance_floor=.01,\n                            coefficient_error_budget=float(np.max(np.abs(approx-phi))*4),\n                            operator_error_bound=float(2*np.max(np.abs(approx-phi))*np.linalg.norm(g["Lt"],2))))\n    pd.DataFrame(fourier).to_csv(cfg["out"]/"fourier_basis.csv",index=False)\n    # FD accuracy at one fixed coefficient, separate from optimisation.\n    q=data["quotes"][0]; A=assemble(g,hats[0]); ex=sla.expm(-q["T"]*A)@g["v0"]\n    fdrows=[]\n    for steps in ([8,16,32,64] if cfg["profile"]=="smoke" else [32,64,128,256,512,1024,2048]):\n        _,_,v=fd_value_grad(g,hats[0],g["v0"],q["T"],np.eye(g["N"]),np.zeros(g["N"]),np.ones(g["N"]),0.,steps)\n        fdrows.append(dict(steps=steps,price_error_inf=float(np.max(np.abs(v-ex)))))\n    pd.DataFrame(fdrows).to_csv(cfg["out"]/"fd_convergence.csv",index=False)\n    # Quote Jacobian singular values identify unobserved parameter directions.\n    jacrows=[]; v=jnp.asarray(g["v0"]); previous=0.\n    for s,q in enumerate(data["quotes"]):\n        fw,_=solver_functions(n,4,"expm",1,1.,float(q["T"]-previous)); previous=q["T"]\n        W=jnp.asarray(interpolation(g["K"],q["k"])[:,1:-1])\n        J=np.asarray(jax.jacrev(lambda z:W@fw(z,v))(jnp.array(hats[s])))\n        sv=sla.svdvals(J)\n        for j,val in enumerate(sv): jacrows.append(dict(slice=s+1,index=j+1,singular_value=val,condition=float(sv[0]/max(sv[-1],1e-15))))\n        v=fw(jnp.array(hats[s]),v)\n    pd.DataFrame(jacrows).to_csv(cfg["out"]/"identifiability.csv",index=False)\n    # Same out-of-span data: M and regularisation are varied independently.\n    ab=[]\n    if cfg["ablation"]:\n        d=make_dataset("out",5.,101,cfg["ref_n"])\n        for M in [2,4,8]:\n            gm=make_grid(n,M)\n            for lam in [0.,1e-7,1e-5,1e-3]:\n                for method in ["expm","lchs-beta"]:\n                    r=calibrate_model(gm,d,method,cfg,0,lam)\n                    ab.append(evaluate_model(gm,d,r,method,0,lam))\n                    pd.DataFrame(ab).to_csv(cfg["out"]/"ablation.csv",index=False)\n    return pd.DataFrame(rows)\n\nMARKET_COLUMNS=["date","expiry","underlying","strike","forward","discount","bid","ask",\n                "option_type","exercise_style","carry_model","source"]\n\ndef load_market_csv(path):\n    frame=pd.read_csv(path)\n    missing=set(MARKET_COLUMNS)-set(frame.columns)\n    if missing: raise ValueError("Missing market columns: "+str(sorted(missing)))\n    for c in ["date","expiry"]: frame[c]=pd.to_datetime(frame[c],errors="raise")\n    for c in ["strike","forward","discount","bid","ask"]:\n        frame[c]=pd.to_numeric(frame[c],errors="raise")\n        if not np.isfinite(frame[c]).all(): raise ValueError("Non-finite "+c)\n    if not (frame.exercise_style.str.lower()=="european").all(): raise ValueError("Only European quotes are supported")\n    if not (frame.carry_model=="deterministic_proportional").all():\n        raise ValueError("Declare deterministic_proportional carry; discrete cash dividends are unsupported")\n    if ((frame[["strike","forward","discount"]]<=0).any().any() or (frame.bid<0).any() or (frame.ask<frame.bid).any()):\n        raise ValueError("Invalid prices, discount, forward or crossed bid/ask")\n    if frame.duplicated(["date","expiry","underlying","strike"]).any():\n        raise ValueError("Duplicate contract/strike rows. Resolve explicitly; do not average silently")\n    if frame.source.fillna("").str.strip().eq("").any(): raise ValueError("A data source is required on every row")\n    datasets=[]\n    for (date,underlying),day in frame.groupby(["date","underlying"]):\n        qs=[]; tests=[]\n        for expiry,part in day.groupby("expiry"):\n            part=part.sort_values("strike").copy()\n            T=(expiry-date).days/365.\n            if T<=0: raise ValueError("Expired or same-day quotes")\n            if len(part)<6: raise ValueError("Need at least six quotes per expiry for held-out evaluation")\n            if part.forward.nunique()!=1 or part.discount.nunique()!=1: raise ValueError("Inconsistent forward/discount per expiry")\n            norm=part.forward.to_numpy()*part.discount.to_numpy()\n            parity=part.discount.to_numpy()*(part.forward.to_numpy()-part.strike.to_numpy())\n            cp=part.option_type.str.lower()\n            if not cp.isin(["call","put"]).all(): raise ValueError("option_type must be call or put")\n            bid=(part.bid.to_numpy()+np.where(cp=="put",parity,0))/norm\n            ask=(part.ask.to_numpy()+np.where(cp=="put",parity,0))/norm\n            k=part.strike.to_numpy()/part.forward.to_numpy(); mid=(bid+ask)/2\n            if k.min()<np.exp(-1.2) or k.max()>np.exp(1.2): raise ValueError("Market strikes exceed the supported domain")\n            held=np.arange(len(k))%4==1  # deterministic split before any fitting\n            qs.append(dict(T=T,k=k[~held],mid=mid[~held],bid=bid[~held],ask=ask[~held]))\n            tests.append(dict(T=T,k=k[held],price=mid[held],bid=bid[held],ask=ask[held]))\n        if len(qs)<2: raise ValueError("Need at least two expiries per market date")\n        ident=hashlib.sha256(f"{date}-{underlying}".encode()).hexdigest()[:12]\n        datasets.append(dict(dataset_id="market_"+ident,kind="market",truth="market",noise_bp=0.,noise_seed=0,\n                             date=str(date.date()),underlying=str(underlying),quotes=qs,tests=tests))\n    return datasets\n\ndef run_market(cfg):\n    path=cfg.get("market_csv",""); out=cfg["out"]\n    if not path:\n        pd.DataFrame(columns=MARKET_COLUMNS).to_csv(out/"market_template.csv",index=False)\n        save_json(out/"market_status.json",dict(status="not_run",reason="No real market CSV supplied; template has headers only"))\n        print("Market study: NOT RUN. Set MARKET_CSV to a bid-ask CSV. No synthetic substitute is used.")\n        return pd.DataFrame()\n    datasets=load_market_csv(path); g=make_grid(cfg["ns"][min(1,len(cfg["ns"])-1)])\n    rows=[]; failed=[]\n    for d in datasets:\n        save_json(out/"raw"/(d["dataset_id"]+".json"),d)\n        for method in ["expm","fd","lchs-beta","AH","PINN-Wam"]:\n            try:\n                if method=="AH": r=ah_calibrate(g,d,cfg)\n                elif method=="PINN-Wam": r=pinn_calibrate(g,d,cfg,0,True)\n                else: r=calibrate_model(g,d,method,cfg,0,1e-7)\n                row=evaluate_model(g,d,r,method,0,1e-7); row.update(date=d["date"],kind="market")\n                hits=[]\n                for s,test in enumerate(d["tests"]):\n                    p=(np.asarray(r["test_predictions"][s]) if "test_predictions" in r\n                       else interpolation(g["K"],test["k"])@r["C"][s])\n                    hits.extend((p>=test["bid"])&(p<=test["ask"]))\n                row["heldout_bidask_fraction"]=float(np.mean(hits)); rows.append(row)\n                save_json(out/"checkpoints"/(d["dataset_id"]+"_"+method+".json"),dict(row=row,result=r))\n            except (ValueError,RuntimeError,FloatingPointError) as exc:\n                failed.append(dict(dataset=d["dataset_id"],method=method,reason=str(exc)))\n        pd.DataFrame(rows).to_csv(out/"market_results.csv",index=False)\n    save_json(out/"market_status.json",dict(status="completed" if rows else "failed",dates=len(datasets),\n              source_sha256=hashlib.sha256(Path(path).read_bytes()).hexdigest(),failures=failed))\n    return pd.DataFrame(rows)\n', 'quantum_checks.py': '"""Explicit small circuits. Dense SELECT synthesis is costed, not called efficient.\n\nNo objective values are loaded into the circuit. The residual is constructed\nfrom the LCHS propagator, the quote observation matrix and reference prices.\n"""\nfrom qiskit import QuantumCircuit, transpile\nfrom qiskit.circuit.library import StatePreparation, UnitaryGate\nfrom qiskit.quantum_info import Statevector\n\ndef contraction_dilation(B):\n    """Halmos unitary dilation; B is a contraction. Ancilla is the high bit."""\n    U,s,Vh=sla.svd(B); s=np.clip(s,0,1)\n    left=(U*np.sqrt(np.maximum(0,1-s*s)))@U.conj().T\n    V=Vh.conj().T; right=(V*np.sqrt(np.maximum(0,1-s*s)))@V.conj().T\n    full=np.block([[B,left],[right,-B.conj().T]])\n    if not np.allclose(full.conj().T@full,np.eye(len(full)),atol=2e-10):\n        raise AssertionError("Dilation is not unitary")\n    return full\n\ndef select_condition(circuit,gate,controls,targets,value):\n    """Little-endian integer convention, explicitly tested by statevector readback."""\n    if not controls:\n        circuit.append(gate,targets); return\n    for j,bit in enumerate(controls):\n        if not ((value>>j)&1): circuit.x(bit)\n    circuit.append(gate.control(len(controls)),controls+targets)\n    for j,bit in enumerate(controls):\n        if not ((value>>j)&1): circuit.x(bit)\n\ndef reflection_success(width,ancillas):\n    """2 P_success - I, with success defined by all listed ancillas being zero."""\n    qc=QuantumCircuit(width,name="2P-I"); qc.global_phase=np.pi\n    for a in ancillas: qc.x(a)\n    if len(ancillas)==1: qc.z(ancillas[0])\n    else: qc.mcp(np.pi,ancillas[:-1],ancillas[-1])\n    for a in ancillas: qc.x(a)\n    return qc\n\ndef build_residual_circuit(cfg):\n    n=cfg["circuit_n"]; m=cfg["circuit_m"]; g=make_grid(n,2)\n    N=g["N"]; d=2*m; D=2**d; J=cfg["circuit_terms"]; b=int(np.log2(J))\n    if 2**b!=J: raise ValueError("Circuit LCU count must be a power of two")\n    k,w,meta=lchs_weights("beta",J,4.); alpha=float(np.abs(w).sum())\n    dt=float(TIMES[0]); c=g["cbar"]\n    design=list(range(d)); work=list(range(d,d+n)); kr=list(range(d+n,d+n+b))\n    wa=d+n+b; branch=wa+1; width=branch+1\n    # Three distinct physical quote locations (two when N=4).\n    qk=np.array([.85,1.1]) if N==4 else np.array([.8,1.,1.25])\n    Wfull=interpolation(g["K"],qk); Wsmall=Wfull[:,1:-1]\n    W=np.zeros((N,N)); W[:len(qk)]=Wsmall\n    gammaW=max(1.,sla.norm(W,2)); UW=contraction_dilation(W/gammaW)\n    xi_star=np.array([.7,.25]); exact=sla.expm(-dt*assemble(g,xi_star))@g["v0"]\n    ref=Wsmall@exact; y=np.zeros(N); y[:len(ref)]=ref\n    vnorm=sla.norm(g["v0"]); ynorm=sla.norm(y)\n    a=vnorm*np.exp(c*dt)*alpha*gammaW; scale=a+ynorm\n    prep=StatePreparation(np.sqrt(np.abs(w)/alpha))\n    forward=QuantumCircuit(width-1,name="LCHS-W")\n    forward.append(StatePreparation(g["v0"]/vnorm),work)\n    forward.append(prep,kr)\n    values=[]; losses=[]; forward_errors=[]\n    for index in range(D):\n        xi=np.array([index%(2**m),index//(2**m)])/(2**m-1)\n        A=assemble(g,xi); L=(A+A.T)/2+c*np.eye(N); H=-.5j*(A-A.T)\n        approx=np.zeros((N,N),complex)\n        for j in range(J):\n            U=sla.expm(-1j*dt*(H+k[j]*L))\n            approx+=w[j]*U\n            gate=UnitaryGate(np.exp(1j*np.angle(w[j]))*U,label="exp(-i(H+kL)t)")\n            # controls ordered design then LCU index\n            select_condition(forward,gate,design+kr,work,index+(j<<d))\n        pv=np.exp(c*dt)*approx@g["v0"]\n        loss=float(np.linalg.norm(W@pv-y)**2)\n        values.append(xi); losses.append(loss)\n        forward_errors.append(float(np.max(np.abs(pv-sla.expm(-dt*A)@g["v0"]))))\n    forward.append(prep.inverse(),kr)\n    forward.append(UnitaryGate(UW,label="quote W dilation"),work+[wa])\n    V=QuantumCircuit(width,name="Residual")\n    branchprep=StatePreparation([np.sqrt(a/scale),np.sqrt(ynorm/scale)])\n    V.append(branchprep,[branch])\n    V.x(branch); V.append(forward.to_gate().control(1),[branch]+list(range(width-1))); V.x(branch)\n    V.append(StatePreparation(y/ynorm).control(1),[branch]+work)\n    V.z(branch)\n    V.append(branchprep.inverse(),[branch])\n    return dict(V=V,g=g,d=d,D=D,N=N,values=np.array(values),losses=np.array(losses),\n                scale=scale,ancillas=kr+[wa,branch],meta=meta,forward_errors=forward_errors,\n                gammaW=gammaW,lcualpha=alpha,dt=dt)\n\ndef run_quantum_checks(cfg):\n    out=cfg["out"]; print("Constructing explicit coherent residual circuit...")\n    t=time.perf_counter(); built=build_residual_circuit(cfg)\n    V=built["V"]; d=built["d"]; D=built["D"]; N=built["N"]; width=V.num_qubits\n    initial=QuantumCircuit(width)\n    for bit in range(d): initial.h(bit)\n    residual_state=Statevector.from_instruction(initial.compose(V)).data.reshape(-1,D)\n    probabilities=np.sum(np.abs(residual_state[:N])**2,axis=0)*D\n    expected=built["losses"]/built["scale"]**2\n    residual_error=float(np.max(np.abs(probabilities-expected)))\n    assert residual_error<2e-8,"Residual circuit does not match matrix calculation"\n    # V^dagger (2P-I) V is block-encoded by clean workspace. Add one\n    # Hadamard-test flag so the clean block is exactly F / scale^2.\n    R=reflection_success(width,built["ancillas"])\n    U=QuantumCircuit(width,name="2F-1")\n    U.compose(V,inplace=True); U.compose(R,inplace=True); U.compose(V.inverse(),inplace=True)\n    F=QuantumCircuit(width+1,name="Objective BE")\n    flag=width; F.h(flag); F.append(U.to_gate().control(1),[flag]+list(range(width))); F.h(flag)\n    ini=QuantumCircuit(width+1)\n    for bit in range(d): ini.h(bit)\n    state=Statevector.from_instruction(ini.compose(F)).data\n    observed=state[:D]*np.sqrt(D)\n    oracle_error=float(np.max(np.abs(observed-expected)))\n    assert oracle_error<3e-8,"Objective block-encoding mismatch"\n    print("Transpiling dense small-instance oracle into u/cx gates...")\n    compiled=transpile(F,basis_gates=["u","cx"],optimization_level=0,seed_transpiler=17)\n    counts=compiled.count_ops()\n    resource=dict(n=cfg["circuit_n"],M=2,m=cfg["circuit_m"],\n                  design_points=D,lcu_terms=cfg["circuit_terms"],qubits=F.num_qubits,\n                  depth=compiled.depth(),cx=int(counts.get("cx",0)),u=int(counts.get("u",0)),\n                  residual_probability_error=residual_error,objective_block_error=oracle_error,\n                  classical_build_validation_s=time.perf_counter()-t,\n                  scale=built["scale"],alpha_lcu=built["lcualpha"],gamma_W=built["gammaW"],\n                  cost_scope="Explicit dense design-enumerated SELECT; not asymptotic efficient synthesis",\n                  gate_set="u,cx, all-to-all connectivity, no fault-tolerant compilation",hardware_execution=False)\n    save_json(out/"quantum_resources.json",resource)\n    frame=pd.DataFrame(dict(index=np.arange(D),xi1=built["values"][:,0],xi2=built["values"][:,1],\n                  physical_loss=built["losses"],normalised_loss=expected,\n                  circuit_probability=probabilities,circuit_block_real=observed.real,\n                  circuit_block_imag=observed.imag,forward_price_error=built["forward_errors"]))\n    frame.to_csv(out/"quantum_objective.csv",index=False)\n    # Save a readable circuit diagram of the high-level validated composition.\n    (out/"diagnostics"/"circuit.txt").write_text(str(F.draw(output="text",fold=100)))\n    print("Quantum statevector checks PASS:",resource)\n    return frame,resource\n\ndef run_qhd(cfg,landscape):\n    """Classical simulation of QHD on the verified discrete objective landscape.\n    The landscape is classically tabulated. This is not a full coherent run.\n    """\n    size=int(np.sqrt(len(landscape))); F=landscape.physical_loss.to_numpy()\n    scale=max(float(np.ptp(F)),1e-12); V=(F-F.min())/scale\n    h=1/(size-1)\n    lap=np.diag(np.r_[1.,np.full(size-2,2.),1.])-np.diag(np.ones(size-1),1)-np.diag(np.ones(size-1),-1)\n    kinetic=(np.kron(np.eye(size),lap)+np.kron(lap,np.eye(size)))/(2*h*h)\n    eig,Q=sla.eigh(kinetic); psi=np.ones(size*size,complex)/size\n    steps=cfg["qhd_steps"]; total=5.; dt=total/steps; records=[]\n    target=F<=F.min()+.01*scale\n    def kinetic_step(p,coef): return Q@(np.exp(-1j*coef*eig)*(Q.T@p))\n    for j in range(steps+1):\n        if j%max(1,steps//100)==0 or j==steps:\n            p=np.abs(psi)**2\n            records.append(dict(step=j,time=j*dt,expected_loss=float(p@F),\n                                best_grid_loss=float(F.min()),near_optimal_probability=float(p@target),norm=float(p.sum())))\n        if j==steps: break\n        s=(j+.5)*dt\n        psi=kinetic_step(psi,.5*dt*np.exp(-s))\n        psi*=np.exp(-1j*dt*np.exp(s)*V)\n        psi=kinetic_step(psi,.5*dt*np.exp(-s))\n    assert abs(np.sum(np.abs(psi)**2)-1)<1e-8\n    result=pd.DataFrame(records); result.to_csv(cfg["out"]/"qhd_trajectory.csv",index=False)\n    final=landscape.copy(); final["probability"]=np.abs(psi)**2\n    final.to_csv(cfg["out"]/"qhd_distribution.csv",index=False)\n    save_json(cfg["out"]/"qhd_status.json",dict(scope="Classically tabulated landscape; split-step QHD simulation",\n            strong_convexity_verified=False,global_continuous_optimum_verified=False,\n            reference="Exact enumeration of this finite design grid",schedule="exp(-s) kinetic + exp(s) normalised potential"))\n    return result\n', 'extended_checks.py': '"""Sensitivity analyses and QHD resolution checks; require preceding cells."""\n\ndef run_domain_slice_sensitivity(cfg):\n    if not cfg["ablation"]:\n        save_json(cfg["out"]/"sensitivity_status.json",dict(status="skipped_smoke_profile"))\n        return pd.DataFrame()\n    rows=[]; n=cfg["ns"][1]; gm=make_grid(n)\n    for S in [2,4,8]:\n        times=np.unique(np.r_[np.arange(1,S+1)/S,float(TIMES[0])])\n        K,C=reference_surface("in",cfg["ref_n"],tuple(times))\n        qs=[]; tests=[]; ks=TRAIN_K[0]; kt=np.linspace(.705,1.395,151)\n        kt=kt[~np.any(np.isclose(kt[:,None],ks[None,:],atol=1e-10,rtol=0),axis=1)]\n        for T,p in zip(times,C):\n            mid=interpolation(K,ks)@p\n            qs.append(dict(T=float(T),k=ks,mid=mid,bid=mid,ask=mid))\n            tests.append(dict(T=float(T),k=kt,price=interpolation(K,kt)@p))\n        data=dict(dataset_id=f"slice_{len(times)}",kind="synthetic",truth="in",noise_bp=0.,noise_seed=101,quotes=qs,tests=tests)\n        for method in ["expm","lchs-beta"]:\n            result=calibrate_model(gm,data,method,cfg,0,0.)\n            row=evaluate_model(gm,data,result,method,0,0.)\n            # The original two-slice coefficient-error indexing does not apply.\n            row.pop("sigma2_linf",None)\n            row.update(test="maturity_slices",nominal_slices=S,actual_slices=len(times),total_quotes=sum(len(q["k"]) for q in qs))\n            rows.append(row)\n            pd.DataFrame(rows).to_csv(cfg["out"]/"domain_slice_sensitivity.csv",index=False)\n    data=make_dataset("in",0.,101,cfg["ref_n"])\n    for domain in [1.,1.2,1.6]:\n        # Fixed N means h changes: report it rather than calling this pure\n        # boundary error. Independent fine-grid domain checks follow.\n        g=make_grid(n,4,-domain,domain)\n        r=calibrate_model(g,data,"expm",cfg,0,0.)\n        row=evaluate_model(g,data,r,"expm",0,0.)\n        row.update(test="fitted_domain",domain_halfwidth=domain,h=g["h"])\n        rows.append(row)\n    pts=np.linspace(.7,1.4,101)\n    K0,C0=reference_surface("in",cfg["ref_n"]+1,tuple(TIMES),-1.6,1.6)\n    for domain in [1.,1.2]:\n        K,C=reference_surface("in",cfg["ref_n"],tuple(TIMES),-domain,domain)\n        err=float(np.max(np.abs(C@interpolation(K,pts).T-C0@interpolation(K0,pts).T)))\n        rows.append(dict(test="reference_domain",domain_halfwidth=domain,reference_domain_price_inf=err,\n                         note="Includes finite reference discretisation error"))\n    frame=pd.DataFrame(rows); frame.to_csv(cfg["out"]/"domain_slice_sensitivity.csv",index=False)\n    save_json(cfg["out"]/"sensitivity_status.json",dict(status="completed"))\n    return frame\n\ndef qhd_landscape_and_refinement(cfg,circuit_landscape):\n    """Denser CLASSICAL tabulation of the same finite-LCHS objective.\n    The actual circuit validates its coarse design points only. The denser\n    landscape does not imply a coherent multi-bit experiment was performed.\n    """\n    size={"smoke":4,"pilot":8,"paper":16}[cfg["profile"]]\n    g=make_grid(cfg["circuit_n"],2); dt=float(TIMES[0])\n    qk=np.array([.85,1.1])\n    W=interpolation(g["K"],qk)[:,1:-1]\n    y=W@(sla.expm(-dt*assemble(g,[.7,.25]))@g["v0"])\n    k,w,_=lchs_weights("beta",cfg["circuit_terms"],4.)\n    rows=[]\n    for j in range(size):\n        for i in range(size):\n            xi=np.array([i,j])/(size-1); A=assemble(g,xi)\n            L=(A+A.T)/2+g["cbar"]*np.eye(g["N"]); H=-.5j*(A-A.T)\n            p=sum(wl*(sla.expm(-1j*dt*(H+kl*L))@g["v0"]) for kl,wl in zip(k,w))*np.exp(g["cbar"]*dt)\n            rows.append(dict(xi1=xi[0],xi2=xi[1],physical_loss=float(np.linalg.norm(W@p-y)**2)))\n    landscape=pd.DataFrame(rows)\n    for row in circuit_landscape.itertuples():\n        match=landscape[(np.isclose(landscape.xi1,row.xi1))&(np.isclose(landscape.xi2,row.xi2))]\n        assert len(match)==1 and abs(match.physical_loss.iloc[0]-row.physical_loss)<1e-12\n    landscape.to_csv(cfg["out"]/"qhd_classical_landscape.csv",index=False)\n    trajectory=run_qhd(cfg,landscape)\n    fine=dict(cfg); fine["out"]=cfg["out"]/"diagnostics"/"qhd_refined"\n    fine["out"].mkdir(parents=True,exist_ok=True); fine["qhd_steps"]=2*cfg["qhd_steps"]\n    refined=run_qhd(fine,landscape)\n    p=pd.read_csv(cfg["out"]/"qhd_distribution.csv").probability.to_numpy()\n    q=pd.read_csv(fine["out"]/"qhd_distribution.csv").probability.to_numpy()\n    row=dict(steps=cfg["qhd_steps"],refined_steps=fine["qhd_steps"],design_points=size*size,\n             probability_total_variation=float(.5*np.abs(p-q).sum()),\n             expected_loss_difference=float(abs(trajectory.expected_loss.iloc[-1]-refined.expected_loss.iloc[-1])),\n             scope="Classical tabulation; coarse circuit points cross-checked; not full coherent QHD")\n    pd.DataFrame([row]).to_csv(cfg["out"]/"qhd_refinement.csv",index=False)\n    return trajectory\n', 'reporting.py': '"""Publication figures and evidence-only LaTeX tables; no simulated results supplied."""\nimport zipfile, shutil\n\nCOLORS={"expm":"#1764A0","fd":"#55813B","lchs-beta":"#D16A22","lchs-simple":"#8A619A",\n        "AH":"#4B5961","PINN-fixed":"#918267","PINN-Wam":"#008D91"}\nLABELS={"expm":"Exact expm","fd":"FD + adjoint","lchs-beta":"LCHS beta",\n        "lchs-simple":"LCHS simple","AH":"Andreasen-Huge","PINN-fixed":"PINN fixed","PINN-Wam":"PINN adaptive"}\n\ndef figure_style():\n    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":8,"axes.titlesize":8,\n        "axes.labelsize":8,"xtick.labelsize":7,"ytick.labelsize":7,"legend.fontsize":6.5,\n        "axes.spines.top":False,"axes.spines.right":False,"pdf.fonttype":42,\n        "ps.fonttype":42,"savefig.dpi":220,"axes.axisbelow":True})\n\ndef export_figure(fig,cfg,name,sources,caption):\n    for ax in fig.axes:\n        if ax.get_label()!="<colorbar>": ax.grid(alpha=.16,linewidth=.5)\n    if cfg["profile"]!="paper":\n        fig.text(.99,-.075,cfg["profile"].upper()+" / implementation check",ha="right",fontsize=6,color="#80562D")\n    fig.savefig(cfg["out"]/"figures"/(name+".pdf"),bbox_inches="tight",pad_inches=.06)\n    fig.savefig(cfg["out"]/"figures"/(name+".png"),bbox_inches="tight",pad_inches=.06)\n    save_json(cfg["out"]/"figures"/(name+".json"),dict(sources=sources,caption=caption,profile=cfg["profile"]))\n    plt.close(fig)\n\ndef make_figures(cfg,results,selected):\n    figure_style(); out=cfg["out"]; n=cfg["ns"][min(1,len(cfg["ns"])-1)]; g=make_grid(n)\n    fig,ax=plt.subplots(2,2,figsize=(3.5,3.45),layout="constrained")\n    for r,truth in enumerate(["in","out"]):\n        data=make_dataset(truth,0.,101,cfg["ref_n"]); s=1\n        plotk=np.linspace(.7,1.4,401)\n        ax[r,0].plot(plotk,true_sigma2(np.log(plotk),s,truth),color="black",lw=1.3,label="Truth")\n        for method in ["expm","lchs-beta"]:\n            key=f\'{data["dataset_id"]}_n{n}_{method}_s{cfg["seeds"][0]}\'\n            if key not in selected: continue\n            result=selected[key]["result"]\n            ax[r,0].plot(plotk,.01+np.asarray(result["hats"][s])@basis(np.log(plotk),g["M"]),color=COLORS[method],lw=1,label=LABELS[method])\n            test=data["tests"][s]; p=interpolation(g["K"],test["k"])@np.asarray(result["C"])[s]\n            ax[r,1].plot(test["k"],(p-test["price"])*1e4,color=COLORS[method],lw=1,label=LABELS[method]+" raw")\n            if method=="lchs-beta":\n                px=interpolation(g["K"],test["k"])@np.asarray(result["C_exact"])[s]\n                ax[r,1].plot(test["k"],(px-test["price"])*1e4,color=COLORS[method],ls=":",lw=1,label="Beta exact re-eval.")\n        ax[r,0].set(xlim=(.7,1.4),ylabel="Local variance",title=("In-family" if truth=="in" else "Out-of-family")+", T=1")\n        ax[r,1].axhline(0,color=".6",lw=.6); ax[r,1].set(ylabel="Price error (bp)",title="Held-out strikes, T=1")\n        for a in ax[r]: a.set_xlabel("Normalised strike k")\n    ax[0,0].legend(loc="best"); ax[0,1].legend(loc="best")\n    export_figure(fig,cfg,"calibration",["selected_examples.json","raw/"],"Seed 0, zero noise, second maturity; explicit dataset identifiers in selected_examples.json.")\n\n    df=pd.read_csv(out/"error_budget.csv")\n    fig,axs=plt.subplots(1,2,figsize=(3.5,2.15),layout="constrained")\n    for kernel,color in [("simple",COLORS["lchs-simple"]),("beta",COLORS["lchs-beta"])]:\n        part=df[(df.n==n)&(df.kernel==kernel)]\n        for _,p in part.groupby("cutoff"):\n            axs[0].loglog(p.terms,p.price_error_inf,":",color=color,alpha=.25,lw=.7)\n        best=part.loc[part.groupby("terms").price_error_inf.idxmin()].sort_values("terms")\n        axs[0].loglog(best.terms,best.price_error_inf,"o-",ms=3,color=color,label=kernel)\n        axs[1].loglog(part.solver_s*1000,part.price_error_inf,"o",ms=3,color=color,label=kernel)\n    axs[0].set(xlabel="LCU terms J",ylabel="Max price error",title=f"Cutoff envelope, N={2**n}")\n    from matplotlib.ticker import NullFormatter\n    axs[0].set_xticks(cfg["kernels"],[str(x) for x in cfg["kernels"]])\n    axs[0].xaxis.set_minor_formatter(NullFormatter())\n    timing=df[df.n==n].solver_s.to_numpy()*1000\n    ticks=np.geomspace(timing.min(),timing.max(),3)\n    axs[1].set_xticks(ticks,[f"{x:.2g}" for x in ticks]); axs[1].xaxis.set_minor_formatter(NullFormatter())\n    axs[1].set(xlabel="Warm evaluation (ms)",ylabel="Max price error",title="Classical simulation")\n    axs[0].legend(); axs[1].legend()\n    export_figure(fig,cfg,"kernel_budget",["error_budget.csv"],"Faint lines show every cutoff; envelope minimises reference error and is not a deployable tuning rule.")\n\n    fig,axs=plt.subplots(1,3,figsize=(3.5,1.85),layout="constrained")\n    for ax,(kind,label) in zip(axs,[("butterfly","Butterfly"),("calendar","Calendar"),("density","Density")]):\n        x=np.maximum(df[kind+"_bound"].to_numpy(),1e-16); y=np.maximum(df[kind+"_violation"].to_numpy(),1e-16)\n        ax.loglog(x,y,"o",ms=2.3,color=COLORS["lchs-beta"],alpha=.6)\n        low=min(x.min(),y.min()); high=max(x.max(),y.max()); ax.plot([low,high],[low,high],"--",color=".5",lw=.7)\n        ax.set(title=label,xlabel="Bound",ylabel="Violation" if kind=="butterfly" else "")\n    export_figure(fig,cfg,"error_certificate",["error_budget.csv"],"Zero violations plotted at 1e-16. Units are price for butterfly/calendar, price per strike squared for density. All bounds are conditional on measured exact-model price error.")\n\n    fig,ax=plt.subplots(figsize=(5.5,3.2),layout="constrained")\n    part=results[(results.n==n)&(results.truth=="out")&(results.noise_bp==5)]\n    methods=[m for m in LABELS if m in set(part.method)]\n    for i,m in enumerate(methods):\n        values=part[part.method==m].rmse_heldout.to_numpy()*1e4\n        ax.scatter(values,i+np.linspace(-.1,.1,len(values)),color=COLORS[m],s=13,alpha=.7)\n    ax.set_yticks(range(len(methods)),[LABELS[m] for m in methods]); ax.invert_yaxis()\n    ax.set(xlabel="Held-out price RMSE (normalised bp)",title=f"Out-of-family, 5 bp noise, N={2**n}")\n    export_figure(fig,cfg,"benchmark_variability",["results.csv"],"Each dot is an executed run; initialisation seeds are not independent market observations.")\n\n    q=pd.read_csv(out/"quantisation.csv"); f=pd.read_csv(out/"fourier_basis.csv")\n    fig,axs=plt.subplots(1,2,figsize=(5.5,2.4),layout="constrained")\n    axs[0].semilogy(q.bits,q.rmse_quotes,"o-",color=COLORS["expm"],label="Quote RMSE")\n    axs[0].semilogy(q.bits,q.rmse_heldout,"s-",color=COLORS["lchs-beta"],label="Held-out RMSE")\n    axs[0].axhline(q.continuous_reference.iloc[0],ls=":",color=".4",label="Continuous quote RMSE")\n    axs[0].set(xlabel="Bits per coefficient",ylabel="Price RMSE",title="Rounded fitted coefficients"); axs[0].legend()\n    axs[1].loglog(f.degree,f.error_inf,"o-",color=COLORS["expm"],label="Basis sup error")\n    axs[1].loglog(f.degree,np.maximum(-f.minimum_basis,1e-18),"s--",color=COLORS["lchs-beta"],label="Negative basis overshoot")\n    axs[1].set(xlabel="Fourier degree",ylabel="Absolute basis error",title="Compact C1 basis"); axs[1].legend()\n    export_figure(fig,cfg,"precision_basis",["quantisation.csv","fourier_basis.csv"],"Quantisation does not re-optimise the discrete objective. Fourier approximation is diagnosed separately from exact pointwise calibration.")\n\n    q=pd.read_csv(out/"quantum_objective.csv")\n    fig,axs=plt.subplots(1,2,figsize=(3.5,1.85),layout="constrained")\n    axs[0].plot(q.normalised_loss,q.circuit_block_real,"o",ms=4,color=COLORS["expm"])\n    lo=q.normalised_loss.min(); hi=q.normalised_loss.max(); axs[0].plot([lo,hi],[lo,hi],"--",color=".5",lw=.7)\n    axs[0].set(xlabel="Matrix objective / scale²",ylabel="Circuit clean block",title="All coherent designs")\n    err=np.abs(q.circuit_block_real+1j*q.circuit_block_imag-q.normalised_loss)\n    axs[1].semilogy(q.index,np.maximum(err,1e-16),"o-",color=COLORS["lchs-beta"],ms=3)\n    axs[1].set(xlabel="Design index",ylabel="Block error",title="Statevector validation")\n    export_figure(fig,cfg,"quantum_validation",["quantum_objective.csv","quantum_resources.json"],"Actual residual/objective circuit, one maturity and two design parameters; dense enumerated SELECT.")\n\n    q=pd.read_csv(out/"qhd_trajectory.csv"); d=pd.read_csv(out/"qhd_distribution.csv")\n    fig,axs=plt.subplots(1,2,figsize=(5.5,2.4),layout="constrained")\n    axs[0].plot(q.time,q.near_optimal_probability,color=COLORS["expm"])\n    axs[0].axhline(q.near_optimal_probability.iloc[0],ls=":",color=".5",label="Uniform initial state")\n    axs[0].set(xlabel="Schedule time",ylabel="Near-grid-optimum probability",ylim=(0,1),title="Classical QHD simulation"); axs[0].legend()\n    sz=int(np.sqrt(len(d))); im=axs[1].imshow(d.probability.to_numpy().reshape(sz,sz),origin="lower",extent=[0,1,0,1],aspect="auto",cmap="Blues")\n    axs[1].set(xlabel="Coefficient 1",ylabel="Coefficient 2",title="Final design probability"); fig.colorbar(im,ax=axs[1],label="Probability")\n    export_figure(fig,cfg,"qhd_diagnostics",["qhd_trajectory.csv","qhd_distribution.csv","qhd_refinement.csv"],"Classically tabulated denser landscape; only the separate coarse design circuit is coherently verified. Near optimal = within 1% of the finite objective range.")\n\n    f=pd.read_csv(out/"fd_convergence.csv"); j=pd.read_csv(out/"identifiability.csv")\n    fig,axs=plt.subplots(1,2,figsize=(5.5,2.3),layout="constrained")\n    axs[0].loglog(f.steps,f.price_error_inf,"o-",color=COLORS["fd"]); axs[0].set(xlabel="Backward Euler steps",ylabel="Max price error",title="Time discretisation at fixed design")\n    for s,p in j.groupby("slice"): axs[1].semilogy(p["index"],p.singular_value,"o-",label=f"Slice {s}")\n    axs[1].set(xlabel="Singular-value index",ylabel="Quote-Jacobian singular value",title="Local identifiability"); axs[1].legend()\n    export_figure(fig,cfg,"fd_identifiability",["fd_convergence.csv","identifiability.csv"],"FD convergence separates time error from optimiser error. Jacobian spectra are local diagnostics.")\n\n    fig,axs=plt.subplots(1,2,figsize=(5.5,2.3),layout="constrained")\n    for method in ["PINN-fixed","PINN-Wam"]:\n        key=f\'out_bp5_d101_n{n}_{method}_s{cfg["seeds"][0]}\'\n        if key not in selected: continue\n        hist=pd.DataFrame(selected[key]["result"]["pinn_history"])\n        for ax,metric in zip(axs,["fit","pde"]): ax.semilogy(hist.step,np.maximum(hist[metric],1e-18),label=LABELS[method],color=COLORS[method])\n    for ax,metric in zip(axs,["Unweighted fit MSE","Unweighted PDE MSE"]):\n        ax.set(xlabel="Training step",ylabel=metric,title="Out-of-family / seed 0"); ax.legend()\n    export_figure(fig,cfg,"pinn_losses",["selected_examples.json"],"Fixed/adaptive network losses; adaptation is not an exact reproduction of the cited study.")\n\n    if (out/"ablation.csv").exists():\n        ab=pd.read_csv(out/"ablation.csv"); fig,axs=plt.subplots(1,2,figsize=(5.5,2.3),layout="constrained")\n        for ax,method in zip(axs,["expm","lchs-beta"]):\n            p=ab[ab.method==method].pivot(index="M",columns="lam",values="rmse_heldout")\n            im=ax.imshow(p.to_numpy()*1e4,aspect="auto",cmap="Blues",vmin=ab.rmse_heldout.min()*1e4,vmax=ab.rmse_heldout.max()*1e4)\n            ax.set_xticks(range(len(p.columns)),[f"{x:g}" for x in p.columns],rotation=30)\n            ax.set_yticks(range(len(p.index)),p.index); ax.set(xlabel="Regularisation lambda",ylabel="Basis size M",title=LABELS[method]); fig.colorbar(im,ax=ax,label="Held-out RMSE (bp)")\n        export_figure(fig,cfg,"ablation",["ablation.csv"],"Out-of-family, fixed 5 bp noise dataset and initialisation. Not a hyperparameter selection study.")\n    if (out/"domain_slice_sensitivity.csv").exists():\n        p=pd.read_csv(out/"domain_slice_sensitivity.csv"); fig,axs=plt.subplots(1,2,figsize=(5.5,2.3),layout="constrained")\n        for method,d in p[p.test=="maturity_slices"].groupby("method"):\n            axs[0].plot(d.actual_slices,d.rmse_heldout*1e4,"o-",label=LABELS[method],color=COLORS[method])\n        d=p[p.test=="fitted_domain"]; axs[1].plot(d.domain_halfwidth,d.rmse_heldout*1e4,"o-",color=COLORS["expm"])\n        axs[0].set(xlabel="Actual maturity slices",ylabel="Held-out RMSE (bp)",title="Extra expiries add observations"); axs[0].legend()\n        axs[1].set(xlabel="Log-strike domain half-width",ylabel="Held-out RMSE (bp)",title="Fixed N: boundary and grid effects")\n        export_figure(fig,cfg,"domain_slices",["domain_slice_sensitivity.csv"],"The true 182/365 breakpoint is retained. More expiries add data; fixed N domain changes also alter h.")\n\ndef tex_escape(s):\n    return str(s).replace("\\\\",r"\\textbackslash{}").replace("_",r"\\_").replace("%",r"\\%").replace("&",r"\\&").replace("#",r"\\#")\n\ndef tex_number(x):\n    if not np.isfinite(x): return "--"\n    return f"{x:.3g}"\n\ndef write_tables(cfg,results):\n    out=cfg["out"]; t=out/"tables"; n=cfg["ns"][min(1,len(cfg["ns"])-1)]\n    failures=json.loads((out/"failures.json").read_text())\n    status=json.loads((out/"market_status.json").read_text())\n    market_status=status["status"]\n    macro=("\\\\renewcommand{\\\\StudyProfile}{"+cfg["profile"]+"}\\n"+\n           "\\\\renewcommand{\\\\RunCount}{"+str(len(results))+"}\\n"+\n           "\\\\renewcommand{\\\\MarketStatus}{"+tex_escape(market_status)+"}\\n"+\n           "\\\\renewcommand{\\\\RunStatus}{"+str(len(failures))+" failed runs are retained in the failure log.}\\n")\n    (t/"run_macros.tex").write_text(macro)\n    # Dataset means prevent initialisation replication from becoming fake data n.\n    part=results[(results.n==n)&(results.truth=="out")&(results.noise_bp==5)]\n    cols=["rmse_quotes","rmse_heldout","iv_rmse","opt_s","butterfly_violation","calendar_violation"]\n    means=part.groupby(["method","dataset"],as_index=False)[cols].mean()\n    abstract=""\n    if cfg["profile"]=="paper":\n        vals={m:float(d.rmse_heldout.mean()*1e4) for m,d in means.groupby("method")}\n        if all(m in vals for m in ["expm","fd","lchs-beta","AH"]):\n            abstract=(f"For the out-of-family, 5-bp-noise case at $N={2**n}$, mean held-out price errors are "\n                f"{vals[\'lchs-beta\']:.3g}, {vals[\'expm\']:.3g}, {vals[\'fd\']:.3g}, and {vals[\'AH\']:.3g} normalised bp "\n                "for beta-LCHS, exact propagation, finite differences, and Andreasen--Huge, respectively.")\n    with (t/"run_macros.tex").open("a") as handle:\n        handle.write("\\\\renewcommand{\\\\AbstractEvidence}{"+abstract+"}\\n")\n    means.to_csv(out/"dataset_means.csv",index=False)\n    lines=[r"\\begin{table}[t]\\centering\\caption{Out-of-family, 5 bp noise, $N="+str(2**n)+r"$. Means over datasets after averaging initialisations. Price errors are normalised bp; time is classical optimisation seconds.}",\n           r"\\label{tab:bench}\\scriptsize\\begin{tabular}{lrrrr}\\toprule Method & Fit & Test & Seconds & Runs\\\\\\midrule"]\n    for method in LABELS:\n        d=means[means.method==method]\n        if d.empty: continue\n        lines.append(tex_escape(LABELS[method])+" & "+" & ".join([tex_number(d.rmse_quotes.mean()*1e4),tex_number(d.rmse_heldout.mean()*1e4),tex_number(d.opt_s.mean()),str(int((part.method==method).sum()))])+r"\\\\")\n    lines += [r"\\bottomrule\\end{tabular}\\end{table}"]\n    (t/"benchmark.tex").write_text("\\n".join(lines))\n    # Full uncertainty summaries retain both hierarchy levels in CSV.\n    metrics=[c for c in cols+["sigma2_linf","compile_warm_s","iv_valid","iv_total"] if c in results]\n    groups=["truth","noise_bp","n","method"]\n    results.groupby(groups)[metrics].agg(["mean","std","min","max","count"]).to_csv(out/"all_run_summary.csv")\n    dataset=results.groupby(groups+["dataset"])[metrics].mean()\n    dataset.groupby(groups)[metrics].agg(["mean","std","count"]).to_csv(out/"between_dataset_summary.csv")\n    results.groupby(groups+["dataset"])[metrics].std().to_csv(out/"within_dataset_initialisation_sd.csv")\n    lc=results[results.method.str.startswith("lchs")]\n    exact=results[results.method=="expm"]\n    summary=(f"The executed {tex_escape(cfg[\'profile\'])} profile contains {len(results)} completed synthetic calibration runs on "\n       f"{results.dataset.nunique()} distinct datasets; {len(failures)} failures are logged. "\n       f"Raw LCHS price-butterfly and calendar violation maxima are {lc.butterfly_violation.max():.3g} and {lc.calendar_violation.max():.3g}, respectively. "\n       f"The exact-exponential price-butterfly violation maximum is {exact.butterfly_violation.max():.3g}. "\n       "These are normalised price quantities, not derivative tolerances. The complete per-run CSV retains non-convergence, invalid-IV counts, compilation costs, and exact re-evaluation audits. ")\n    if cfg["profile"]!="paper": summary+="These reduced-budget results are implementation evidence only and must not be presented as the full study. "\n    summary+="Between-dataset standard deviations and within-dataset initialisation standard deviations are exported separately; a single dataset does not support a between-dataset uncertainty estimate.\\n"\n    (t/"results_summary.tex").write_text(summary)\n    resource=json.loads((out/"quantum_resources.json").read_text())\n    text=(f"The executed one-slice circuit uses {resource[\'qubits\']} qubits and {resource[\'design_points\']} coherent design points. "\n          f"The maximum clean-block error is {resource[\'objective_block_error\']:.3g}. "\n          f"Dense synthesis produces {resource[\'cx\']:,} CNOT gates, {resource[\'u\']:,} single-qubit gates, and depth {resource[\'depth\']:,}. "\n          "These counts use the declared all-to-all u/CNOT basis and generic design enumeration; they are not a fault-tolerant resource estimate.\\n")\n    (t/"quantum_summary.tex").write_text(text)\n    if market_status=="not_run":\n        mt="Real-market study: not run. No market observations were supplied. The input template and validation/held-out evaluation code are provided; no synthetic result is labelled as market evidence.\\n"\n    else:\n        market=pd.read_csv(out/"market_results.csv")\n        mt=(f"The market input contains {status.get(\'dates\',0)} date/underlying datasets with {len(market)} completed method runs and {len(status.get(\'failures\',[]))} failures. "\n            "The source checksum, raw normalised snapshots, held-out bid--ask coverage and full method results accompany the manuscript. ")\n        if len(market):\n            mt+=r"\\begin{table}[t]\\centering\\caption{Executed real-market runs: held-out price error and bid--ask coverage.}\\scriptsize\\begin{tabular}{lrr}\\toprule Method & RMSE (bp) & Coverage\\\\\\midrule"+"\\n"\n            for m,d in market.groupby("method"):\n                mt+=tex_escape(LABELS[m])+f" & {1e4*d.rmse_heldout.mean():.3g} & {d.heldout_bidask_fraction.mean():.3f}"+r"\\\\"+"\\n"\n            mt+=r"\\bottomrule\\end{tabular}\\end{table}"\n    (t/"market_summary.tex").write_text(mt)\n    save_json(out/"evidence_status.json",dict(profile=cfg["profile"],completed_runs=len(results),failures=len(failures),\n             market_status=market_status,quantum_hardware_executed=False,full_coherent_qhd_executed=False,\n             evidence_ready_for_author_review=cfg["profile"]=="paper" and market_status=="completed"))\n\ndef make_supplement(cfg):\n    out=cfg["out"]\n    intro=r"""\\documentclass[10pt]{article}\n\\usepackage[margin=20mm]{geometry}\n\\usepackage{graphicx,booktabs,amsmath}\n\\usepackage[hidelinks]{hyperref}\n\\title{Supplementary diagnostics: local-volatility calibration}\n\\author{Anonymous submission}\n\\date{}\n\\begin{document}\\maketitle\nThis supplement is generated from executed experiments. Profile: PROFILE.\nSmoke and pilot outputs are validation/exploration, not final study evidence.\nEvery figure has a same-name JSON source record. Full metrics, unsuccessful\noptimisation runs, failed baseline calls, and invalid implied-volatility counts\nare retained in the accompanying CSV/JSON files.\n""".replace("PROFILE",cfg["profile"])\n    names=[("benchmark_variability","All executed initialisations, out-of-family noisy case."),\n           ("precision_basis","Coefficient rounding and global Fourier approximation."),\n           ("fd_identifiability","Time-discretisation and local identifiability diagnostics."),\n           ("pinn_losses","Documented fixed/adaptive PINN comparison."),\n           ("qhd_diagnostics",r"Classically tabulated QHD simulation; see qhd\\_refinement.csv."),\n           ("ablation","Independent basis-size and regularisation ablation."),\n           ("domain_slices","Maturity and finite-domain sensitivity, with explicit confounders.")]\n    for name,caption in names:\n        if (out/"figures"/(name+".pdf")).exists():\n            intro+=r"\\begin{figure}[htbp]\\centering\\includegraphics[width=.92\\linewidth]{figures/"+name+r".pdf}\\caption{"+caption+r"}\\end{figure}"+"\\n"\n    intro+=r"\\end{document}"\n    (out/"supplement.tex").write_text(intro)\n\ndef finalise_outputs(cfg,results,selected,main_tex,readme,source_files):\n    make_figures(cfg,results,selected); write_tables(cfg,results); make_supplement(cfg)\n    out=cfg["out"]; (out/"main.tex").write_text(main_tex,encoding="utf-8")\n    (out/"README.md").write_text(readme,encoding="utf-8")\n    (out/"source").mkdir(exist_ok=True)\n    for name,content in source_files.items(): (out/"source"/name).write_text(content,encoding="utf-8")\n    manifest=json.loads((out/"manifest.json").read_text()); manifest["study_status"]="completed"\n    manifest["market_status"]=json.loads((out/"market_status.json").read_text())["status"]\n    manifest["source_sha256"]={name:hashlib.sha256(content.encode()).hexdigest() for name,content in source_files.items()}\n    manifest["manuscript_sha256"]=hashlib.sha256(main_tex.encode()).hexdigest()\n    save_json(out/"manifest.json",manifest)\n    archive=out.parent/(out.name+"_results.zip")\n    with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.rglob("*")):\n            if p.is_file() and p.suffix not in [".zip",".tmp"]: z.write(p,p.relative_to(out))\n    print("Generated",len(list((out/"figures").glob("*.pdf"))),"figure PDFs, LaTeX tables, manuscript and supplement.")\n    print("Result archive:",archive)\n    return archive\n'}

## 13. Generate all figures, LaTeX tables and downloadable results
The tables report only executed metrics. All approximate outputs are raw and kept separate from
exact re-evaluations. Open the PNG previews below; the PDF figures are generated in the same folder.


In [ ]:
archive=finalise_outputs(cfg,results,selected,MAIN_TEX,README_TEXT,SOURCE_FILES)
from IPython.display import FileLink, Image, display
for name in ["calibration","kernel_budget","error_certificate","quantum_validation"]:
    display(Image(filename=str(cfg["out"]/"figures"/(name+".png")),width=750))
display(FileLink(str(archive)))
print("In Colab, the optional cell below downloads this result ZIP.")


In [ ]:
DOWNLOAD_ZIP = False  # Set True and rerun this cell to download in Colab.
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(archive))


## Interpretation
`evidence_status.json` states which parts ran. Confirm full `paper` execution, reference and time-step
refinement, optimiser failures, baseline budgets, and real-market evidence before drawing conclusions.
No quantum advantage is claimed. Package versions, data checksums,
run identifiers, source hashes, and all figure source records are preserved in the ZIP.
